In [ ]:
# IMPORTANT: SOME KAGGLE DATA SOURCES ARE PRIVATE
# RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES.
import kagglehub
kagglehub.login()


In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.

abhilash437_md17_ethanol_path = kagglehub.dataset_download('abhilash437/md17-ethanol')
abhilash437_md17_aspirin_path = kagglehub.dataset_download('abhilash437/rmd17-aspirin')
abhilash437_md17_malonaldehyde_path = kagglehub.dataset_download('abhilash437/rmd17-malonaldehyde')
abhilash437_anchored_block_diagonal_koopman_model_pytorch_default_1_path = kagglehub.model_download('abhilash437/anchored-block-diagonal-koopman-model/PyTorch/default/1')
abhilash437_gru_baseline_pytorch_default_1_path = kagglehub.model_download('abhilash437/gru-baseline/PyTorch/default/1')

print('Data source import complete.')


Data source import complete.


In [ ]:
%pip install torch_geometric statsmodels -q

import os
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import Adam
from torch.utils.data import DataLoader, TensorDataset
import torch_geometric as pyg
from torch_geometric.nn import GCNConv
from torch_geometric.data import Data
from torch_geometric.utils import to_undirected, add_remaining_self_loops

import numpy as np
import scipy.sparse as sp
from scipy.stats import ttest_ind
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import connected_components
from sklearn.metrics import roc_auc_score, f1_score

import matplotlib.pyplot as plt
import seaborn as sns
import json
import time
from pathlib import Path
from tqdm import tqdm
from sklearn.metrics import f1_score, precision_score, recall_score
import pandas as pd

import torch.optim as optim

# Device
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {DEVICE}")

PROJECT_ROOT = "/kaggle/working/koopman_md17_project"
DIRS = {
    "data": os.path.join(PROJECT_ROOT, "data"),
    "checkpoints": os.path.join(PROJECT_ROOT, "checkpoints"),
    "results": os.path.join(PROJECT_ROOT, "results")
}

# Safely generate the directory tree
for dir_name, dir_path in DIRS.items():
    os.makedirs(dir_path, exist_ok=True)
    print(f"Directory established: {dir_path}")

# Device
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {DEVICE}")

# Random seed for reproducibility
RANDOM_SEED = 42
torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

Device: cpu
Directory established: /kaggle/working/koopman_md17_project/data
Directory established: /kaggle/working/koopman_md17_project/checkpoints
Directory established: /kaggle/working/koopman_md17_project/results
Device: cpu


In [ ]:
"""
dataset_adapter
──────────────────
Base interface and MD17 adapter for the Koopman dynamics framework.

Usage
-----
adapter = MD17Adapter(path="/kaggle/working/.../md17_ethanol.npz",
                      molecule="ethanol",
                      sub_sampling=2,
                      window_len=150,
                      train_frac=0.8)

train_split, test_split = adapter.load()

# train_split.X      : np.ndarray  (N_train, T-1, D*2)
# train_split.y      : np.ndarray  (N_train,)   int labels
# train_split.lengths: List[int]   true lengths before padding
# train_split.meta   : dict        logging / plot titles
"""

from abc import ABC, abstractmethod
from dataclasses import dataclass, field
from typing import Tuple, List

import numpy as np


# ─────────────────────────────────────────────────────────────────────────────
# 1. Standard data container
# ─────────────────────────────────────────────────────────────────────────────

@dataclass
class DatasetSplit:
    """
    Standard container produced by every adapter.

    Fields
    ------
    X       np.ndarray  shape (N, T, D)
                N = number of trajectory windows
                T = max sequence length across the split (shorter ones padded
                    with zeros to this length)
                D = feature dimension per timestep

    y       np.ndarray  shape (N,)  int64
                Class label per trajectory.
                If the dataset has no meaningful labels, this is all zeros.

    lengths List[int]   length N
                True (unpadded) length of each trajectory.
                The model uses these to ignore the zero-padded tail.

    meta    dict
                Dataset-specific metadata for logging and plot titles.
                Guaranteed keys: 'dataset', 'molecule', 'D', 'split'
    """
    X      : np.ndarray
    y      : np.ndarray
    lengths: List[int]
    meta   : dict = field(default_factory=dict)


# ─────────────────────────────────────────────────────────────────────────────
# 2. Abstract base class — every new dataset subclasses this
# ─────────────────────────────────────────────────────────────────────────────

class DynamicsDatasetAdapter(ABC):
    """
    Contract all dataset adapters must satisfy.

    The training loop and evaluator only call:
        train_split, test_split = adapter.load()
        model = SomeModel(input_dim=adapter.input_dim, ...)

    They never touch raw files, windowing logic, or label encodings.
    """

    @abstractmethod
    def load(self) -> Tuple[DatasetSplit, DatasetSplit]:
        """
        Parse raw data, compute features, window, pad, split.
        Returns (train_split, test_split).
        """
        pass

    @property
    @abstractmethod
    def input_dim(self) -> int:
        """
        Feature dimension D seen by the model.

        Must be computable BEFORE load() is called so the model can be
        instantiated before loading data into memory.
        """
        pass

    @property
    @abstractmethod
    def name(self) -> str:
        """Human-readable dataset name used in log lines and plot titles."""
        pass

In [ ]:
"""
dataset_adapter.py
──────────────────
Base interface and MD17 adapter for the Koopman dynamics framework.

Usage
-----
adapter = MD17Adapter(path="/kaggle/working/.../md17_ethanol.npz",
                      molecule="ethanol",
                      sub_sampling=2,
                      window_len=150,
                      train_frac=0.8)

train_split, test_split = adapter.load()

# train_split.X      : np.ndarray  (N_train, T-1, D*2)
# train_split.y      : np.ndarray  (N_train,)   int labels
# train_split.lengths: List[int]   true lengths before padding
# train_split.meta   : dict        logging / plot titles
"""

from abc import ABC, abstractmethod
from dataclasses import dataclass, field
from typing import Tuple, List

import numpy as np


# ─────────────────────────────────────────────────────────────────────────────
# 1. Standard data container
# ─────────────────────────────────────────────────────────────────────────────

@dataclass
class DatasetSplit:
    """
    Standard container produced by every adapter.

    Fields
    ------
    X       np.ndarray  shape (N, T, D)
                N = number of trajectory windows
                T = max sequence length across the split (shorter ones padded
                    with zeros to this length)
                D = feature dimension per timestep

    y       np.ndarray  shape (N,)  int64
                Class label per trajectory.
                If the dataset has no meaningful labels, this is all zeros.

    lengths List[int]   length N
                True (unpadded) length of each trajectory.
                The model uses these to ignore the zero-padded tail.

    meta    dict
                Dataset-specific metadata for logging and plot titles.
                Guaranteed keys: 'dataset', 'molecule', 'D', 'split'
    """
    X      : np.ndarray
    y      : np.ndarray
    lengths: List[int]
    meta   : dict = field(default_factory=dict)


# ─────────────────────────────────────────────────────────────────────────────
# 2. Abstract base class — every new dataset subclasses this
# ─────────────────────────────────────────────────────────────────────────────

class DynamicsDatasetAdapter(ABC):
    """
    Contract all dataset adapters must satisfy.

    The training loop and evaluator only call:
        train_split, test_split = adapter.load()
        model = SomeModel(input_dim=adapter.input_dim, ...)

    They never touch raw files, windowing logic, or label encodings.
    """

    @abstractmethod
    def load(self) -> Tuple[DatasetSplit, DatasetSplit]:
        """
        Parse raw data, compute features, window, pad, split.
        Returns (train_split, test_split).
        """
        pass

    @property
    @abstractmethod
    def input_dim(self) -> int:
        """
        Feature dimension D seen by the model.

        Must be computable BEFORE load() is called so the model can be
        instantiated before loading data into memory.
        """
        pass

    @property
    @abstractmethod
    def name(self) -> str:
        """Human-readable dataset name used in log lines and plot titles."""
        pass


# ─────────────────────────────────────────────────────────────────────────────
# 3. MD17 adapter
# ─────────────────────────────────────────────────────────────────────────────

class MD17Adapter(DynamicsDatasetAdapter):
    """
    Adapter for the MD17 molecular dynamics benchmark.

    Raw file format
    ---------------
    .npz with keys: 'R' (coordinates), 'E' (energies), optionally 'F' (forces)
        R : (n_frames, n_atoms, 3)   atomic positions in Angstrom
        E : (n_frames,)              potential energy in kcal/mol

    Feature construction (mirrors the original notebook pipeline)
    ─────────────────────────────────────────────────────────────
    Step 1 — Sub-sample:  take every `sub_sampling`-th frame from R and E.

    Step 2 — Pairwise distances:  for each frame, compute the upper triangle
             of the pairwise Euclidean distance matrix.
             flat_dim = n_atoms * (n_atoms - 1) // 2
             This gives rotational and translational invariance.

    Step 3 — Window:  cut the sub-sampled trajectory into non-overlapping
             segments of length `window_len`.  Each segment becomes one
             trajectory in the dataset.

    Step 4 — Label:  each window is labeled 0 (low energy) or 1 (high energy)
             based on whether its mean energy is above the median.
             This matches the original notebook convention.

    Step 5 — Phase-space velocity:  inside build_phase_space() below,
             velocity at time t is defined as the finite difference
                 v_t = x_{t+1} - x_t
             and the feature at time t is [x_{t+1} || v_t].
             This consumes one frame, so the output length is window_len - 1.

    Step 6 — Pad:  within each split, pad all trajectories to the same length
             with zeros so they can be stacked into a single ndarray.

    Parameters
    ----------
    path         : str   path to the .npz file
    molecule     : str   molecule name, used only in metadata
    sub_sampling : int   take every N-th frame  (default 2)
    window_len   : int   frames per trajectory window  (default 150)
    train_frac   : float fraction of windows used for training  (default 0.8)
    """

    # atom counts for common MD17 molecules — used by input_dim
    _ATOM_COUNTS = {
        "aspirin"      : 21,
        "benzene"      : 12,
        "ethanol"      :  9,
        "malonaldehyde": 9,
        "naphthalene"  : 18,
        "salicylic"    : 16,
        "toluene"      : 15,
        "uracil"       : 12,
    }

    def __init__(
        self,
        path        : str,
        molecule    : str  = "ethanol",
        sub_sampling: int  = 2,
        window_len  : int  = 150,
        train_frac  : float = 0.8,
    ):
        self.path         = path
        self.molecule     = molecule.lower()
        self.sub_sampling = sub_sampling
        self.window_len   = window_len
        self.train_frac   = train_frac

        # Resolve n_atoms — try the lookup table first, fall back to the file
        if self.molecule in self._ATOM_COUNTS:
            self._n_atoms = self._ATOM_COUNTS[self.molecule]
        else:
            # Will be set properly during load()
            self._n_atoms = None

    # ── Public properties ────────────────────────────────────────────────────

    @property
    def name(self) -> str:
        return f"MD17-{self.molecule}"

    @property
    def input_dim(self) -> int:
        """
        D = 2 * flat_dim  because each timestep = [position || velocity]
        and both have the same shape (upper triangle of distance matrix).

        flat_dim = n_atoms * (n_atoms - 1) // 2
        """
        if self._n_atoms is None:
            raise RuntimeError(
                "n_atoms unknown for molecule '{}'. "
                "Call load() first or add it to _ATOM_COUNTS.".format(self.molecule)
            )
        flat_dim = self._n_atoms * (self._n_atoms - 1) // 2
        return 2 * flat_dim

    # ── Main entry point ─────────────────────────────────────────────────────

    def load(self) -> Tuple[DatasetSplit, DatasetSplit]:
        """
        Full pipeline: raw file → (train_split, test_split).
        """

        # ── Step 1: read raw file ────────────────────────────────────────────
        # Supports both rMD17 ('coords'/'energies') and
        # older MD17 ('R'/'E') key conventions.
        raw = np.load(self.path, allow_pickle=True)

        coords   = raw["coords"]   if "coords"   in raw else raw["R"]
        energies = raw["energies"] if "energies" in raw else raw["E"]

        # rMD17 stores n_atoms directly in nuclear_charges
        if "nuclear_charges" in raw:
            self._n_atoms = int(raw["nuclear_charges"].shape[0])

        n_frames, n_atoms, _ = coords.shape
        if self._n_atoms is None:
            self._n_atoms = n_atoms    # fallback if not set from nuclear_charges

        # ── Step 2: sub-sample ───────────────────────────────────────────────
        coords   = coords[::self.sub_sampling]
        energies = energies[::self.sub_sampling]

        # ── Step 3: pairwise distances ───────────────────────────────────────
        #
        # For each frame, compute the upper triangle of the n_atoms x n_atoms
        # Euclidean distance matrix.
        #
        # diff[i,j] = coord_i - coord_j  →  shape (n_frames, n_atoms, n_atoms, 3)
        # We use vectorized broadcasting across all frames at once to avoid
        # a Python loop over frames, which would be slow for large datasets.
        #
        diff = coords[:, :, None, :] - coords[:, None, :, :]
        # diff shape: (n_frames, n_atoms, n_atoms, 3)

        dist_matrix = np.sqrt(np.sum(diff ** 2, axis=-1))
        # dist_matrix shape: (n_frames, n_atoms, n_atoms)

        triu_idx  = np.triu_indices(n_atoms, k=1)
        flat_dist = dist_matrix[:, triu_idx[0], triu_idx[1]]
        # flat_dist shape: (n_frames, flat_dim)
        # flat_dim = n_atoms * (n_atoms - 1) // 2

        # ── Step 4: window + label ───────────────────────────────────────────
        trajectories, labels = self._make_windows(flat_dist, energies)

        # ── Step 5: train / test split ───────────────────────────────────────
        split_idx = int(len(trajectories) * self.train_frac)
        train_trajs  = trajectories[:split_idx]
        train_labels = labels[:split_idx]
        test_trajs   = trajectories[split_idx:]
        test_labels  = labels[split_idx:]

        # ── Step 6: phase-space velocity + pad ──────────────────────────────
        train_split = self._build_split(train_trajs, train_labels, split="train")
        test_split  = self._build_split(test_trajs,  test_labels,  split="test")

        return train_split, test_split

    # ── Private helpers ──────────────────────────────────────────────────────

    def _make_windows(
        self,
        flat_dist: np.ndarray,   # (n_frames, flat_dim)
        energies : np.ndarray,   # (n_frames,)
    ) -> Tuple[List[np.ndarray], List[int]]:
        """
        Cut flat_dist into non-overlapping windows of length window_len.
        Label each window 1 (high energy) or 0 (low energy) relative to
        the global median — matching the original notebook convention.

        Returns
        -------
        trajectories : List of np.ndarray, each shape (window_len, flat_dim)
        labels       : List of int  (0 or 1)
        """
        total_steps = flat_dist.shape[0]
        # Integer division — the tail that does not fill a complete window
        # is discarded, matching the original notebook.
        n_windows   = (total_steps - self.window_len) // self.window_len

        threshold    = np.median(energies)
        trajectories = []
        labels       = []

        for idx in range(n_windows):
            start = idx * self.window_len
            end   = start + self.window_len

            trajectories.append(flat_dist[start:end])   # (window_len, flat_dim)
            mean_energy = np.mean(energies[start:end])
            labels.append(1 if mean_energy > threshold else 0)

        return trajectories, labels

    def _build_split(
        self,
        trajectories: List[np.ndarray],   # each (window_len, flat_dim)
        labels      : List[int],
        split       : str,
    ) -> DatasetSplit:
        """
        Apply phase-space velocity construction and zero-pad to a fixed
        length within this split.

        Phase-space step
        ─────────────────
        For trajectory of shape (T, flat_dim):
            position at step t  = traj[t+1]          shape (flat_dim,)
            velocity at step t  = traj[t+1] - traj[t] shape (flat_dim,)
            feature at step t   = [position || velocity]  shape (2*flat_dim,)

        This produces a sequence of length T-1 = window_len - 1.
        The -1 is why lengths stores window_len - 1, not window_len.

        Padding
        ────────
        Within the split, all sequences are padded with zeros to max_len
        (= window_len - 1, which is the same for every window since all
        windows have the same length).  The padding logic is written to
        handle variable-length sequences too — this will matter when we
        add datasets with variable trajectory lengths.
        """
        phase_sequences = []
        true_lengths    = []

        for traj in trajectories:
            # traj: (window_len, flat_dim)
            pos = traj[1:]          # (window_len-1, flat_dim)  — aligned positions
            vel = traj[1:] - traj[:-1]  # (window_len-1, flat_dim)  — finite diff velocity
            phase = np.concatenate([pos, vel], axis=-1)  # (window_len-1, 2*flat_dim)
            phase_sequences.append(phase)
            true_lengths.append(phase.shape[0])

        # Pad to max length within this split
        max_len  = max(true_lengths)
        flat_dim = trajectories[0].shape[1]
        D        = 2 * flat_dim

        X_padded = np.zeros((len(phase_sequences), max_len, D), dtype=np.float32)
        for i, seq in enumerate(phase_sequences):
            X_padded[i, :seq.shape[0], :] = seq

        y = np.array(labels, dtype=np.int64)

        meta = {
            "dataset"      : "MD17",
            "molecule"     : self.molecule,
            "D"            : D,
            "flat_dim"     : flat_dim,
            "n_atoms"      : self._n_atoms,
            "window_len"   : self.window_len,
            "sub_sampling" : self.sub_sampling,
            "split"        : split,
            "n_trajectories": len(phase_sequences),
        }

        return DatasetSplit(
            X       = X_padded,
            y       = y,
            lengths = true_lengths,
            meta    = meta,
        )

In [ ]:
import os

adapter = MD17Adapter(
    path         = os.path.join(abhilash437_md17_ethanol_path, 'rmd17_ethanol.npz'),
    molecule     = "ethanol",
    sub_sampling = 2,
    window_len   = 150,
    train_frac   = 0.8,
)

train_split, test_split = adapter.load()

# These replace your current X_mol_train, y_mol_train, etc.
X_mol_train  = train_split.X
y_mol_train  = train_split.y
# lengths are inside train_split.lengths — pass to build_velocity_batch as before

print(adapter.input_dim)   # should print 72 for ethanol
print(train_split.meta)

72
{'dataset': 'MD17', 'molecule': 'ethanol', 'D': 72, 'flat_dim': 36, 'n_atoms': 9, 'window_len': 150, 'sub_sampling': 2, 'split': 'train', 'n_trajectories': 265}


In [ ]:
# Quick sanity check after loading
train_split, test_split = adapter.load()

print(f"input_dim      : {adapter.input_dim}")          # expect 72
print(f"train X shape  : {train_split.X.shape}")        # (N_train, 149, 72)
print(f"test  X shape  : {test_split.X.shape}")         # (N_test,  149, 72)
print(f"train lengths  : {set(train_split.lengths)}")   # should be {149}
print(f"meta           : {train_split.meta}")

input_dim      : 72
train X shape  : (265, 149, 72)
test  X shape  : (67, 149, 72)
train lengths  : {149}
meta           : {'dataset': 'MD17', 'molecule': 'ethanol', 'D': 72, 'flat_dim': 36, 'n_atoms': 9, 'window_len': 150, 'sub_sampling': 2, 'split': 'train', 'n_trajectories': 265}


In [ ]:
# ── Part 1: Simplified Model Definitions ─────────────────────────────────────
import torch
import torch.nn as nn
import torch.nn.functional as F

class SimpleKoopmanNet(nn.Module):
    """
    Koopman dynamics model with a single encoder and orthogonal transition
    operator parameterized via the skew-symmetric exponential map.

    K = exp(A - A^T)  is orthogonal by construction at every forward pass.
    No post-step SVD projection needed. No block-diagonal split. No classifier.

    Parameters
    ----------
    input_dim  : int   phase_space_dim (already doubled: position || velocity)
    latent_dim : int   latent space dimension (default 32)
    hidden_dim : int   encoder hidden layer width (default 64)
    """

    def __init__(self, input_dim, latent_dim=32, hidden_dim=64):
        super().__init__()
        self.latent_dim = latent_dim

        self.encoder = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.GELU(),
            nn.LayerNorm(hidden_dim),
            nn.Linear(hidden_dim, latent_dim),
        )

        # Skew-symmetric parameterization: K = exp(A - A^T)
        # Small init keeps K ≈ I at the start of training
        self.A_raw = nn.Parameter(torch.randn(latent_dim, latent_dim) * 0.02)

    @property
    def K(self):
        """Orthogonal transition matrix via exponential map of skew-symmetric A."""
        A_skew = self.A_raw - self.A_raw.T
        return torch.matrix_exp(A_skew)

    def forward(self, x, lengths):
        """
        Encode all frames. No classifier, no rollout — just the trajectory.

        Parameters
        ----------
        x       : (B, T, D) phase-space features
        lengths : list[int] true lengths per sequence

        Returns
        -------
        z_seq : (B, T, latent_dim)
        """
        B, T, D = x.shape
        z_flat = self.encoder(x.reshape(B * T, D))
        z_seq  = z_flat.view(B, T, self.latent_dim)
        return z_seq

    def forward_rollout(self, z0, steps=100, latent_seed=True):
        """
        Autonomous rollout: z_t = z_0 @ K^t^T

        Parameters
        ----------
        z0          : (B, 1, latent_dim) if latent_seed=True, else (B, 1, D)
        steps       : int
        latent_seed : bool — if False, encode z0[:,0] first

        Returns
        -------
        (B, steps, latent_dim)
        """
        if latent_seed:
            z = z0[:, 0]
        else:
            z = self.encoder(z0[:, 0])

        K = self.K
        rollout = [z]
        for t in range(1, steps):
            K_pow = torch.matrix_power(K, t)
            z_t = torch.matmul(z0[:, 0] if latent_seed else z, K_pow.t())
            rollout.append(z_t)
        # Fix: rollout from initial z0, not accumulated
        rollout = []
        z_init = z
        for t in range(steps):
            K_pow = torch.matrix_power(K, t)
            rollout.append(torch.matmul(z_init, K_pow.t()))
        return torch.stack(rollout, dim=1)

    def post_step_hook(self):
        """No-op — orthogonality is guaranteed by the exponential map."""
        pass


class SimpleGRUNet(nn.Module):
    """
    GRU dynamics baseline with the CORRECT autonomous transition:
        z_{t+1} = GRUCell(input=z_t, hidden=z_t)

    NOT GRUCell(input=zeros, hidden=z_t) which degenerates to a fixed-point
    iterator because the cell gates see the same null input at every step.

    Parameters
    ----------
    input_dim  : int   phase_space_dim
    latent_dim : int   default 32
    hidden_dim : int   default 64
    """

    def __init__(self, input_dim, latent_dim=32, hidden_dim=64):
        super().__init__()
        self.latent_dim = latent_dim

        self.encoder = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.GELU(),
            nn.LayerNorm(hidden_dim),
            nn.Linear(hidden_dim, latent_dim),
        )

        self.rnn_transition = nn.GRUCell(latent_dim, latent_dim)

    def forward(self, x, lengths):
        """
        Encode all frames → z_seq.  No classifier.

        Returns
        -------
        z_seq : (B, T, latent_dim)
        """
        B, T, D = x.shape
        z_flat = self.encoder(x.reshape(B * T, D))
        z_seq  = z_flat.view(B, T, self.latent_dim)
        return z_seq

    def forward_rollout(self, z0, steps=100, latent_seed=True):
        """
        Autonomous rollout with correct GRU transition:
            z_{t+1} = GRUCell(input=z_t, hidden=z_t)

        Parameters
        ----------
        z0          : (B, 1, latent_dim) if latent_seed=True
        steps       : int
        latent_seed : bool

        Returns
        -------
        (B, steps, latent_dim)
        """
        if latent_seed:
            z = z0[:, 0]
        else:
            z = self.encoder(z0[:, 0])

        rollout = [z]
        for _ in range(1, steps):
            z = self.rnn_transition(z, z)   # ← z feeds BOTH input and hidden
            rollout.append(z)
        return torch.stack(rollout, dim=1)

    def post_step_hook(self):
        """No-op for GRU."""
        pass


print(f"SimpleKoopmanNet defined — K guaranteed orthogonal via exp(A - A^T)")
print(f"SimpleGRUNet defined — transition uses GRUCell(z, z), not GRUCell(0, z)")


In [ ]:
# ── Part 2: Unified Loss Functions + Simplified Trainer ──────────────────────
import os
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import r2_score

# ─────────────────────────────────────────────────────────────────────────────
# Loss: Koopman
# ─────────────────────────────────────────────────────────────────────────────

def koopman_compute_loss(self, outputs, targets, lengths, epoch):
    """
    Pure dynamics loss for SimpleKoopmanNet.

    z_pred = z_t @ K^T    (one-step linear prediction)
    l_dyn  = MSE(z_pred, z_{t+1})
    l_collapse = relu(0.05 - mean_relative_change)

    targets / epoch are unused — signature kept for Trainer compatibility.
    """
    z_seq = outputs  # (B, T, latent_dim)
    K = self.K       # (latent_dim, latent_dim), orthogonal

    # Collect valid (z_t, z_{t+1}) pairs respecting true lengths
    z_t_list   = []
    z_tgt_list = []
    for b, L in enumerate(lengths):
        if L < 2:
            continue
        z_t_list.append(z_seq[b, :L-1])    # (L-1, D)
        z_tgt_list.append(z_seq[b, 1:L])   # (L-1, D)

    z_t   = torch.cat(z_t_list,   dim=0)   # (N_total, D)
    z_tgt = torch.cat(z_tgt_list, dim=0)   # (N_total, D)

    # One-step prediction via linear operator
    z_pred = torch.matmul(z_t, K.t())       # (N_total, D)

    l_dyn = F.mse_loss(z_pred, z_tgt)

    # Collapse guard: relative change hinge
    step_diff = torch.norm(z_tgt - z_t, dim=-1)           # (N_total,)
    step_norm = torch.norm(z_t, dim=-1) + 1e-8            # (N_total,)
    relative_change = (step_diff / step_norm).mean()
    l_collapse = torch.relu(0.05 - relative_change)

    total_loss = l_dyn + 2.0 * l_collapse

    log = {
        'loss':       total_loss.item(),
        'l_dyn':      l_dyn.item(),
        'l_collapse': l_collapse.item(),
    }
    return total_loss, log


# ─────────────────────────────────────────────────────────────────────────────
# Loss: GRU
# ─────────────────────────────────────────────────────────────────────────────

def gru_compute_loss(self, outputs, targets, lengths, epoch):
    """
    Pure dynamics loss for SimpleGRUNet.

    z_pred = GRUCell(input=z_t, hidden=z_t)   ← correct autonomous step
    l_dyn  = MSE(z_pred, z_{t+1})
    l_collapse = relu(0.05 - mean_relative_change)
    """
    z_seq = outputs  # (B, T, latent_dim)

    z_t_list   = []
    z_tgt_list = []
    for b, L in enumerate(lengths):
        if L < 2:
            continue
        z_t_list.append(z_seq[b, :L-1])
        z_tgt_list.append(z_seq[b, 1:L])

    z_t   = torch.cat(z_t_list,   dim=0)
    z_tgt = torch.cat(z_tgt_list, dim=0)

    # Correct GRU step: z feeds both input and hidden
    z_pred = self.rnn_transition(z_t, z_t)

    l_dyn = F.mse_loss(z_pred, z_tgt)

    # Collapse guard
    step_diff = torch.norm(z_tgt - z_t, dim=-1)
    step_norm = torch.norm(z_t, dim=-1) + 1e-8
    relative_change = (step_diff / step_norm).mean()
    l_collapse = torch.relu(0.05 - relative_change)

    total_loss = l_dyn + 2.0 * l_collapse

    log = {
        'loss':       total_loss.item(),
        'l_dyn':      l_dyn.item(),
        'l_collapse': l_collapse.item(),
    }
    return total_loss, log


# Attach to model classes
SimpleKoopmanNet.compute_loss = koopman_compute_loss
SimpleGRUNet.compute_loss     = gru_compute_loss

print("Unified compute_loss attached to both models.")


# ─────────────────────────────────────────────────────────────────────────────
# Simplified Trainer
# ─────────────────────────────────────────────────────────────────────────────

class SimpleTrainer:
    """
    Dataset-agnostic trainer for pure dynamics models.

    - No classification, no AUC, no composite score.
    - Checkpoints by val_r2 only.
    - Log columns: Epoch | Loss | l_dyn | l_collapse | Val R²

    The model must implement:
        model(x, lengths) → z_seq   (B, T, latent_dim)
        model.compute_loss(z_seq, targets=None, lengths=lengths, epoch=epoch)
        model.post_step_hook()
    """

    def __init__(
        self,
        model,
        optimizer,
        checkpoint_dir,
        checkpoint_name="best.pt",
        epochs=100,
        batch_size=16,
        device="cpu",
        log_every=10,
    ):
        self.model           = model.to(device)
        self.optimizer       = optimizer
        self.checkpoint_dir  = checkpoint_dir
        self.checkpoint_name = checkpoint_name
        self.epochs          = epochs
        self.batch_size      = batch_size
        self.device          = device
        self.log_every       = log_every
        os.makedirs(checkpoint_dir, exist_ok=True)

    def fit(self, train_split, val_split):
        """
        Train loop. Returns best checkpoint info dict.
        """
        X_train = torch.tensor(train_split.X, dtype=torch.float32, device=self.device)
        train_lengths = train_split.lengths

        X_val = torch.tensor(val_split.X, dtype=torch.float32, device=self.device)
        val_lengths = val_split.lengths

        N = X_train.shape[0]
        best_r2   = -1e9
        best_info = {}

        for epoch in range(1, self.epochs + 1):
            # ── Train ────────────────────────────────────────────────────
            avg_log = self._train_epoch(X_train, train_lengths, N, epoch)

            # ── Validate ─────────────────────────────────────────────────
            val_r2 = self._compute_val_r2(X_val, val_lengths)

            # ── Checkpoint ───────────────────────────────────────────────
            if val_r2 > best_r2:
                best_r2 = val_r2
                best_info = {
                    'epoch':  epoch,
                    'val_r2': val_r2,
                }
                torch.save({
                    'epoch':            epoch,
                    'model_state_dict': self.model.state_dict(),
                    'val_r2':           val_r2,
                }, os.path.join(self.checkpoint_dir, self.checkpoint_name))

            # ── Log ──────────────────────────────────────────────────────
            if epoch % self.log_every == 0 or epoch == 1:
                print(
                    f"Epoch {epoch:>3d}/{self.epochs} | "
                    f"Loss {avg_log.get('loss', 0):.4f} | "
                    f"l_dyn {avg_log.get('l_dyn', 0):.4f} | "
                    f"l_coll {avg_log.get('l_collapse', 0):.4f} | "
                    f"Val R² {val_r2:.4f}  "
                    f"{'← BEST' if best_info.get('epoch') == epoch else ''}"
                )

        print(f"\nBest → epoch {best_info['epoch']}, Val R² = {best_info['val_r2']:.4f}")
        return best_info

    def _train_epoch(self, X, lengths, N, epoch):
        self.model.train()
        perm = torch.randperm(N)
        total_log   = {}
        num_batches = 0

        for start in range(0, N, self.batch_size):
            idx = perm[start : start + self.batch_size]
            batch_x    = X[idx]
            batch_lens = [lengths[i] for i in idx.tolist()]

            self.optimizer.zero_grad()
            z_seq = self.model(batch_x, batch_lens)

            loss, log = self.model.compute_loss(
                z_seq, targets=None, lengths=batch_lens, epoch=epoch
            )

            loss.backward()
            nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=1.0)
            self.optimizer.step()
            self.model.post_step_hook()

            for k, v in log.items():
                total_log[k] = total_log.get(k, 0.0) + v
            num_batches += 1

        return {k: v / num_batches for k, v in total_log.items()}

    @torch.no_grad()
    def _compute_val_r2(self, X_val, val_lengths):
        """
        One-step prediction R² on validation set.
        """
        self.model.eval()

        B, T, D = X_val.shape
        z_seq = self.model(X_val, val_lengths)

        z_t_list   = []
        z_tgt_list = []
        for b, L in enumerate(val_lengths):
            if L < 2:
                continue
            z_t_list.append(z_seq[b, :L-1])
            z_tgt_list.append(z_seq[b, 1:L])

        z_t   = torch.cat(z_t_list,   dim=0).cpu().numpy()
        z_tgt = torch.cat(z_tgt_list, dim=0).cpu().numpy()

        if hasattr(self.model, 'K'):
            K = self.model.K.detach().cpu().numpy()
            z_pred = z_t @ K.T
        else:
            z_t_torch = torch.tensor(z_t, dtype=torch.float32, device=self.device)
            z_pred = self.model.rnn_transition(z_t_torch, z_t_torch).cpu().numpy()

        r2 = r2_score(z_tgt.ravel(), z_pred.ravel())
        self.model.train()
        return r2


print("SimpleTrainer defined — checkpoints by Val R², no classification.")


In [ ]:
# ── Part 4: Simplified Evaluator Definition ──────────────────────────────────
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from dataclasses import dataclass, field
from typing import Optional

@dataclass
class SimpleEvalResults:
    """
    Standard evaluation results container for single-encoder models.
    """
    koopman_mse_mean : np.ndarray
    koopman_mse_std  : np.ndarray
    baseline_mse_mean: np.ndarray
    baseline_mse_std : np.ndarray

    rho_koopman      : float

    koopman_geom_mean : np.ndarray
    koopman_geom_std  : np.ndarray
    baseline_geom_mean: np.ndarray
    baseline_geom_std : np.ndarray

    relative_change_koopman : float
    relative_change_baseline: float

    meta: dict = field(default_factory=dict)


class SimpleKoopmanEvaluator:
    """
    Evaluator class for unified, single-encoder Koopman vs GRU models.
    """
    def __init__(
        self,
        koopman_model  : nn.Module,
        baseline_model : nn.Module,
        device         : str = 'cpu',
        rollout_steps  : int = 29,
        batch_size     : int = 64,
    ):
        self.koopman_model  = koopman_model.to(device).eval()
        self.baseline_model = baseline_model.to(device).eval()
        self.device         = device
        self.rollout_steps  = rollout_steps
        self.batch_size     = batch_size

    def run(self, split: DatasetSplit) -> SimpleEvalResults:
        X       = split.X
        lengths = split.lengths
        X_t     = torch.tensor(X, dtype=torch.float32, device=self.device)

        with torch.no_grad():
            z_koop = self._encode_all(self.koopman_model, X_t, lengths)
            z_base = self._encode_all(self.baseline_model, X_t, lengths)

            koop_mse_mean, koop_mse_std = self._rollout_mse(self.koopman_model, z_koop, lengths)
            base_mse_mean, base_mse_std = self._rollout_mse(self.baseline_model, z_base, lengths)

            rho_koopman = self._spectral_radii(self.koopman_model)

            koop_geom_mean, koop_geom_std = self._geometry_retention(self.koopman_model, z_koop, lengths)
            base_geom_norm, base_geom_std = self._geometry_retention(self.baseline_model, z_base, lengths)

            rc_koop = self._relative_change(z_koop, lengths)
            rc_base = self._relative_change(z_base, lengths)

        return SimpleEvalResults(
            koopman_mse_mean        = koop_mse_mean,
            koopman_mse_std         = koop_mse_std,
            baseline_mse_mean       = base_mse_mean,
            baseline_mse_std        = base_mse_std,
            rho_koopman             = rho_koopman,
            koopman_geom_mean       = koop_geom_mean,
            koopman_geom_std        = koop_geom_std,
            baseline_geom_mean      = base_geom_norm,
            baseline_geom_std       = base_geom_std,
            relative_change_koopman = rc_koop,
            relative_change_baseline= rc_base,
            meta                    = split.meta,
        )

    def _encode_all(self, model, X_t, lengths):
        N, T, D = X_t.shape
        x_flat  = X_t.view(N * T, D)
        z_chunks = []
        for start in range(0, N * T, self.batch_size * T):
            end   = min(start + self.batch_size * T, N * T)
            chunk = x_flat[start:end]
            z_chunks.append(model.encoder(chunk))
        return torch.cat(z_chunks, dim=0).view(N, T, -1)

    def _rollout_mse(self, model, z, lengths):
        z_cpu = z.cpu()
        mse_per_step_mean = np.zeros(self.rollout_steps)
        mse_per_step_std  = np.zeros(self.rollout_steps)

        if not hasattr(model, 'K'):
            # GRU path
            for s in range(1, self.rollout_steps + 1):
                errors = []
                for b, T in enumerate(lengths):
                    if T <= s + 1:
                        continue
                    seed = z_cpu[b:b+1, :1, :]
                    preds = model.forward_rollout(
                        seed.to(self.device), steps=s + 1, latent_seed=True
                    ).cpu()
                    z_pred   = preds[0, s]
                    z_target = z_cpu[b, s]
                    errors.append(F.mse_loss(z_pred, z_target).item())
                mse_per_step_mean[s - 1] = np.mean(errors) if errors else 0.0
                mse_per_step_std[s - 1]  = np.std(errors)  if errors else 0.0
        else:
            # Koopman path
            K = model.K.detach().cpu()
            for s in range(1, self.rollout_steps + 1):
                K_pow = torch.matrix_power(K, s)
                errors = []
                for b, T in enumerate(lengths):
                    if T <= s + 1:
                        continue
                    z_init   = z_cpu[b, :T - s]
                    z_target = z_cpu[b, s:T]
                    z_pred   = torch.matmul(z_init, K_pow.t())
                    errors.append(F.mse_loss(z_pred, z_target).item())
                mse_per_step_mean[s - 1] = np.mean(errors) if errors else 0.0
                mse_per_step_std[s - 1]  = np.std(errors)  if errors else 0.0

        return mse_per_step_mean, mse_per_step_std

    def _spectral_radii(self, model):
        if not hasattr(model, 'K'):
            return 0.0
        K = model.K.detach().cpu().numpy()
        eigvals = np.linalg.eigvals(K)
        return float(np.max(np.abs(eigvals)))

    def _geometry_retention(self, model, z, lengths):
        z_cpu  = z.cpu()
        N      = len(lengths)
        n_pairs = min(200, N * (N - 1) // 2)

        rng   = np.random.default_rng(42)
        idx_i = rng.integers(0, N, size=n_pairs)
        idx_j = rng.integers(0, N, size=n_pairs)
        mask  = idx_i != idx_j
        idx_i, idx_j = idx_i[mask], idx_j[mask]

        geom_means = np.zeros(self.rollout_steps + 1)
        geom_stds  = np.zeros(self.rollout_steps + 1)

        with torch.no_grad():
            z_all_0 = z.to(self.device)[:, :1, :]
            rollout_all = model.forward_rollout(z_all_0, steps=self.rollout_steps + 1, latent_seed=True).cpu()

        for s in range(self.rollout_steps + 1):
            ratios = []
            for i, j in zip(idx_i, idx_j):
                T_i = min(lengths[i], lengths[j])
                if T_i <= s:
                    continue
                d0 = torch.norm(rollout_all[i, 0] - rollout_all[j, 0]).item()
                if d0 < 1e-8:
                    continue
                ds = torch.norm(rollout_all[i, s] - rollout_all[j, s]).item()
                ratios.append(ds / d0)

            geom_means[s] = np.mean(ratios) if ratios else 0.0
            geom_stds[s]  = np.std(ratios)  if ratios else 0.0

        return geom_means, geom_stds

    def _relative_change(self, z, lengths):
        z_cpu = z.cpu()
        all_ratios = []
        for b, T in enumerate(lengths):
            if T < 2:
                continue
            z_seq    = z_cpu[b, :T]
            diffs    = (z_seq[1:] - z_seq[:-1]).norm(dim=1)
            norms    = z_seq[:-1].norm(dim=1)
            ratios   = (diffs / (norms + 1e-6)).tolist()
            all_ratios.extend(ratios)
        return float(np.mean(all_ratios)) if all_ratios else 0.0

    def print_summary(self, results: SimpleEvalResults):
        sep = "=" * 70
        print(sep)
        print(f"EVALUATION SUMMARY — {results.meta.get('dataset','?')} {results.meta.get('molecule','')}")
        print(sep)
        print("\n[1] ROLLOUT MSE")
        print(f"  {'Step':>4}  {'Koopman':>10}  {'Baseline':>10}")
        for s in range(len(results.koopman_mse_mean)):
            print(f"  {s+1:>4}  {results.koopman_mse_mean[s]:>10.4f}  {results.baseline_mse_mean[s]:>10.4f}")

        print("\n[2] SPECTRAL RADIUS")
        print(f"  rho(K)      : {results.rho_koopman:.6f}  "
              f"{'PASS (conservative)' if abs(results.rho_koopman - 1.0) < 1e-3 else 'WARN'}")

        print("\n[3] GEOMETRY RETENTION @ final step")
        print(f"  Koopman : {results.koopman_geom_mean[-1]:.4f}")
        print(f"  Baseline: {results.baseline_geom_mean[-1]:.4f}")

        print("\n[4] COLLAPSE DIAGNOSTIC")
        print(f"  Koopman  relative change: {results.relative_change_koopman:.4f}  "
              f"{'PASS' if results.relative_change_koopman > 0.05 else 'FAIL'}")
        print(f"  Baseline relative change: {results.relative_change_baseline:.4f}")
        print(sep)

    def plot(self, results: SimpleEvalResults, title: Optional[str] = None, save_path: Optional[str] = None):
        KOOP_COLOR = "#2166ac"
        BASE_COLOR = "#d6604d"

        steps_err  = np.arange(1, self.rollout_steps + 1)
        steps_geom = np.arange(0, self.rollout_steps + 1)

        k0 = results.koopman_geom_mean[0] + 1e-8
        b0 = results.baseline_geom_mean[0] + 1e-8
        koop_geom_norm = results.koopman_geom_mean  / k0
        base_geom_norm = results.baseline_geom_mean / b0
        koop_gstd_norm = results.koopman_geom_std   / k0
        base_gstd_norm = results.baseline_geom_std  / b0

        dataset_label = (title or f"{results.meta.get('dataset','?')} {results.meta.get('molecule','')}")

        fig = plt.figure(figsize=(13, 8))
        gs  = gridspec.GridSpec(2, 1, hspace=0.3)

        # Top: MSE
        ax1 = fig.add_subplot(gs[0])
        ax1.plot(steps_err, results.koopman_mse_mean, color=KOOP_COLOR, linewidth=2.5, marker='o', markersize=4, label='Koopman (Orthogonal)')
        ax1.fill_between(steps_err, results.koopman_mse_mean - results.koopman_mse_std, results.koopman_mse_mean + results.koopman_mse_std, color=KOOP_COLOR, alpha=0.15)
        ax1.plot(steps_err, results.baseline_mse_mean, color=BASE_COLOR, linewidth=2.5, linestyle='--', marker='x', markersize=5, label='GRU Baseline')
        ax1.fill_between(steps_err, results.baseline_mse_mean - results.baseline_mse_std, results.baseline_mse_mean + results.baseline_mse_std, color=BASE_COLOR, alpha=0.15)
        ax1.set_ylabel("Rollout MSE", fontsize=12)
        ax1.set_title(f"Predictive Accuracy vs. Geometric Stability — {dataset_label}", fontsize=13, fontweight='bold', pad=10)
        ax1.legend(fontsize=11, loc='upper left')
        ax1.grid(True, linestyle=':', alpha=0.5)

        # Bottom: Geometry
        ax2 = fig.add_subplot(gs[1], sharex=ax1)
        ax2.plot(steps_geom, koop_geom_norm, color=KOOP_COLOR, linewidth=2.5, marker='o', markersize=4)
        ax2.fill_between(steps_geom, koop_geom_norm - koop_gstd_norm, koop_geom_norm + koop_gstd_norm, color=KOOP_COLOR, alpha=0.15)
        ax2.plot(steps_geom, base_geom_norm, color=BASE_COLOR, linewidth=2.5, linestyle='--', marker='x', markersize=5)
        ax2.fill_between(steps_geom, base_geom_norm - base_gstd_norm, base_geom_norm + base_gstd_norm, color=BASE_COLOR, alpha=0.15)
        ax2.axhline(1.0, color='gray', linewidth=1.0, linestyle=':', alpha=0.7, label='Perfect retention (ratio = 1.0)')
        ax2.set_xlabel("Prediction Horizon (steps)", fontsize=12)
        ax2.set_ylabel("Pairwise Distance Ratio\n(normalized to t=0)", fontsize=12)
        ax2.legend(fontsize=10, loc='lower left')
        ax2.grid(True, linestyle=':', alpha=0.5)

        if save_path:
            plt.savefig(save_path, dpi=150, bbox_inches='tight')
        plt.show()

print("SimpleKoopmanEvaluator defined successfully.")


In [ ]:
# ── Part 3: Load Data and Train Simple Models on Ethanol ─────────────────────
import os
import torch

# 1. Safe loading check for data splits
if 'train_split' not in globals() or 'test_split' not in globals():
    print("Loading dataset splits...")
    if 'adapter' not in globals():
        # Ensure we have the adapter defined
        adapter = MD17Adapter(
            path         = os.path.join(abhilash437_md17_ethanol_path, 'rmd17_ethanol.npz'),
            molecule     = "ethanol",
            sub_sampling = 2,
            window_len   = 150,
            train_frac   = 0.8,
        )
    train_split, test_split = adapter.load()

# 2. Hyperparameters & Setup
input_dim = adapter.input_dim   # 72 (from md17-ethanol adapter)
latent_dim = 32
hidden_dim = 64

# 3. Instantiate and train SimpleKoopmanNet
print("\nInitializing SimpleKoopmanNet...")
koopman_model_eth = SimpleKoopmanNet(
    input_dim=input_dim,
    latent_dim=latent_dim,
    hidden_dim=hidden_dim
)
koopman_optimizer_eth = torch.optim.AdamW(
    koopman_model_eth.parameters(),
    lr=1e-3,
    weight_decay=1e-4
)

koopman_trainer_eth = SimpleTrainer(
    model           = koopman_model_eth,
    optimizer       = koopman_optimizer_eth,
    checkpoint_dir  = DIRS["checkpoints"],
    checkpoint_name = "simple_koopman_ethanol_best.pt",
    epochs          = 100,
    batch_size      = 16,
    device          = DEVICE,
    log_every       = 10,
)

print("Training SimpleKoopmanNet...")
koopman_best_eth = koopman_trainer_eth.fit(train_split, test_split)


# 4. Instantiate and train SimpleGRUNet
print("\nInitializing SimpleGRUNet...")
gru_model_eth = SimpleGRUNet(
    input_dim=input_dim,
    latent_dim=latent_dim,
    hidden_dim=hidden_dim
)
gru_optimizer_eth = torch.optim.AdamW(
    gru_model_eth.parameters(),
    lr=1e-3,
    weight_decay=1e-4
)

gru_trainer_eth = SimpleTrainer(
    model           = gru_model_eth,
    optimizer       = gru_optimizer_eth,
    checkpoint_dir  = DIRS["checkpoints"],
    checkpoint_name = "simple_gru_ethanol_best.pt",
    epochs          = 100,
    batch_size      = 16,
    device          = DEVICE,
    log_every       = 10,
)

print("Training SimpleGRUNet...")
gru_best_eth = gru_trainer_eth.fit(train_split, test_split)


In [ ]:
# ── Part 5: Ethanol Evaluation & Comparison ──────────────────────────────────
import os

# 1. Reload best checkpoints
koopman_ckpt = torch.load(
    os.path.join(DIRS["checkpoints"], "simple_koopman_ethanol_best.pt"),
    weights_only=False
)
koopman_model_eth.load_state_dict(koopman_ckpt["model_state_dict"])
print(f"Loaded best Koopman checkpoint from epoch {koopman_ckpt['epoch']}")

gru_ckpt = torch.load(
    os.path.join(DIRS["checkpoints"], "simple_gru_ethanol_best.pt"),
    weights_only=False
)
gru_model_eth.load_state_dict(gru_ckpt["model_state_dict"])
print(f"Loaded best GRU checkpoint from epoch {gru_ckpt['epoch']}")

# 2. Run evaluator
evaluator = SimpleKoopmanEvaluator(
    koopman_model  = koopman_model_eth,
    baseline_model = gru_model_eth,
    device         = DEVICE,
    rollout_steps  = 29,
    batch_size     = 64,
)

results = evaluator.run(test_split)

# 3. Print summary and show tradeoff plots
evaluator.print_summary(results)
evaluator.plot(
    results,
    save_path=os.path.join(DIRS["results"], "simple_ethanol_tradeoff.png")
)


In [ ]:
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

# from dataset_adapter import DatasetSplit
from dataclasses import dataclass, field
from typing import Any, Dict, List, Optional


# ─────────────────────────────────────────────────────────────────────────────
# 1. Results container
# ─────────────────────────────────────────────────────────────────────────────

@dataclass
class EvalResults:
    """
    All metrics produced by one evaluator.run() call.

    Metric groups
    ─────────────
    rollout   : MSE at each prediction horizon step (mean ± std over sequences)
    spectral  : spectral radii of the Koopman operator blocks
    geometry  : pairwise latent distance retention over the rollout horizon
    collapse  : relative change metric confirming the encoder is not collapsed
    meta      : passed through from DatasetSplit.meta for logging / plot titles
    """
    # Rollout MSE — shape (rollout_steps,) for mean, same for std
    koopman_mse_mean : np.ndarray
    koopman_mse_std  : np.ndarray
    baseline_mse_mean: np.ndarray
    baseline_mse_std : np.ndarray

    # Spectral radii
    rho_kin  : float
    rho_pheno: float

    # Geometry retention — normalized pairwise distance ratio
    # shape (rollout_steps + 1,) — index 0 is t=0 (always 1.0 after normalization)
    koopman_geom_mean : np.ndarray
    koopman_geom_std  : np.ndarray
    baseline_geom_mean: np.ndarray
    baseline_geom_std : np.ndarray

    # Collapse diagnostic
    relative_change_koopman : float
    relative_change_baseline: float

    # Metadata from the DatasetSplit
    meta: dict = field(default_factory=dict)


# ─────────────────────────────────────────────────────────────────────────────
# 2. Evaluator
# ─────────────────────────────────────────────────────────────────────────────

class KoopmanEvaluator:
    """
    Computes all four metric groups for a Koopman model vs a baseline model
    on any DatasetSplit produced by a DynamicsDatasetAdapter.

    Parameters
    ----------
    koopman_model   : nn.Module
        Must expose:
            - encoder_kin(x_flat)   → z_kin   (N, sub_dim)
            - encoder_pheno(x_flat) → z_pheno (N, sub_dim)
            - K                     → (latent_dim, latent_dim) tensor
              where K[:sub_dim, :sub_dim] is the kinetic block

    baseline_model  : nn.Module
        Must expose:
            - encoder(x_flat) or encoder_shared(x_flat) → z (N, latent_dim)
            - K                                         → (latent_dim, latent_dim)
              (or the model implements forward_rollout for GRU-style baseline)

    device          : str   'cpu' or 'cuda'
    rollout_steps   : int   number of autoregressive steps to evaluate
    sub_dim         : int   size of kinetic subspace (default 16)
    batch_size      : int   sequences processed at a time during encoding
    """

    def __init__(
        self,
        koopman_model  : nn.Module,
        baseline_model : nn.Module,
        device         : str = 'cpu',
        rollout_steps  : int = 29,
        sub_dim        : int = 16,
        batch_size     : int = 64,
    ):
        self.koopman_model  = koopman_model.to(device).eval()
        self.baseline_model = baseline_model.to(device).eval()
        self.device         = device
        self.rollout_steps  = rollout_steps
        self.sub_dim        = sub_dim
        self.batch_size     = batch_size

    # ── Public entry point ───────────────────────────────────────────────────

    def run(self, split: DatasetSplit) -> EvalResults:
        """
        Full evaluation pipeline on one DatasetSplit.
        Returns an EvalResults dataclass.
        """
        X       = split.X        # (N, T, D)  numpy, already phase-space form
        lengths = split.lengths  # List[int]

        # Convert to torch — the evaluator owns the device placement
        X_t = torch.tensor(X, dtype=torch.float32, device=self.device)

        with torch.no_grad():
            z_koop, z_base = self._encode_all(X_t, lengths)
            # z_koop: (N, T, latent_dim)  — cat(z_kin, z_pheno)
            # z_base: (N, T, latent_dim)

            K_koop = self.koopman_model.K.detach().cpu()
            K_base = self._get_baseline_K()

            koop_mse_mean, koop_mse_std   = self._rollout_mse(z_koop, lengths, K_koop)
            base_mse_mean, base_mse_std   = self._rollout_mse(z_base, lengths, K_base)

            rho_kin, rho_pheno            = self._spectral_radii(K_koop)

            koop_geom_mean, koop_geom_std = self._geometry_retention(z_koop, lengths, K_koop)
            base_geom_mean, base_geom_std = self._geometry_retention(z_base, lengths, K_base)

            rc_koop = self._relative_change(z_koop, lengths)
            rc_base = self._relative_change(z_base, lengths)

        return EvalResults(
            koopman_mse_mean        = koop_mse_mean,
            koopman_mse_std         = koop_mse_std,
            baseline_mse_mean       = base_mse_mean,
            baseline_mse_std        = base_mse_std,
            rho_kin                 = rho_kin,
            rho_pheno               = rho_pheno,
            koopman_geom_mean       = koop_geom_mean,
            koopman_geom_std        = koop_geom_std,
            baseline_geom_mean      = base_geom_mean,
            baseline_geom_std       = base_geom_std,
            relative_change_koopman = rc_koop,
            relative_change_baseline= rc_base,
            meta                    = split.meta,
        )

    # ── Encoding ─────────────────────────────────────────────────────────────

    def _encode_all(self, X_t, lengths):
        """
        Encode every frame of every sequence through both models.

        X_t : (N, T, D) torch tensor on self.device
        Returns z_koop (N, T, latent_dim), z_base (N, T, latent_dim)

        Processing in batches of self.batch_size to avoid OOM on large datasets.
        Frames are encoded flat (N*T, D) then reshaped back to (N, T, latent_dim).
        Padding frames are encoded too but ignored downstream via the lengths mask.
        """
        N, T, D = X_t.shape
        x_flat  = X_t.view(N * T, D)   # (N*T, D)

        # Encode in chunks
        z_kin_chunks   = []
        z_pheno_chunks = []
        z_base_chunks  = []

        for start in range(0, N * T, self.batch_size * T):
            end   = min(start + self.batch_size * T, N * T)
            chunk = x_flat[start:end]

            z_kin_chunks.append(self.koopman_model.encoder_kin(chunk))
            z_pheno_chunks.append(self.koopman_model.encoder_pheno(chunk))
            z_base_chunks.append(self._encode_baseline(chunk))

        z_kin   = torch.cat(z_kin_chunks,   dim=0).view(N, T, -1)
        z_pheno = torch.cat(z_pheno_chunks, dim=0).view(N, T, -1)
        z_koop  = torch.cat([z_kin, z_pheno], dim=-1)   # (N, T, latent_dim)
        z_base  = torch.cat(z_base_chunks,  dim=0).view(N, T, -1)

        return z_koop, z_base

    def _encode_baseline(self, x_flat):
        """
        Handles both naming conventions for the baseline encoder:
            BlackBoxGraphDynamicsNet  uses  .encoder
            TrajectoryWindowedKoopmanNet uses .encoder_shared
        """
        if hasattr(self.baseline_model, 'encoder'):
            return self.baseline_model.encoder(x_flat)
        elif hasattr(self.baseline_model, 'encoder_shared'):
            return self.baseline_model.encoder_shared(x_flat)
        else:
            raise AttributeError(
                "baseline_model must have an 'encoder' or 'encoder_shared' attribute."
            )

    def _get_baseline_K(self):
        """
        Retrieves the baseline transition matrix K.
        For GRU-style models that don't have a K attribute, we compute
        an empirical linear approximation via least squares on the encoded
        latent trajectories — this gives a fair spectral comparison.
        """
        if hasattr(self.baseline_model, 'K'):
            return self.baseline_model.K.detach().cpu()
        else:
            # GRU has no K — return None; rollout will use forward_rollout instead
            return None

    # ── Metric: rollout MSE ──────────────────────────────────────────────────

    def _rollout_mse(self, z, lengths, K):
        """
        Autoregressive multi-step rollout MSE.

        z       : (N, T, latent_dim) encoded trajectories
        lengths : List[int]          true lengths per sequence
        K       : (latent_dim, latent_dim) or None (GRU path)

        For each horizon step s in 1..rollout_steps:
            pred_t = z_t @ K^s^T
            error  = MSE(pred_t, z_{t+s})  averaged over all valid (b, t) pairs

        Key term — matrix power K^s:
            Applying K once predicts one step ahead. Applying it s times
            (matrix_power) predicts s steps ahead in one operation, which is
            equivalent to s sequential applications but cheaper to compute.

        Returns (mean_errors, std_errors) each shape (rollout_steps,)
        """
        z_cpu = z.cpu()
        mse_per_step_mean = np.zeros(self.rollout_steps)
        mse_per_step_std  = np.zeros(self.rollout_steps)

        if K is None:
            # GRU path — use forward_rollout if available
            return self._rollout_mse_gru(z, lengths)

        for s in range(1, self.rollout_steps + 1):
            K_pow = torch.matrix_power(K, s)   # (latent_dim, latent_dim)
            errors = []

            for b, T in enumerate(lengths):
                if T <= s + 1:
                    continue
                # Predict z_{t+s} from z_t using K^s
                z_init   = z_cpu[b, :T - s]           # (T-s, latent_dim)
                z_target = z_cpu[b, s:T]               # (T-s, latent_dim)
                z_pred   = torch.matmul(z_init, K_pow.t())

                mse = torch.nn.functional.mse_loss(z_pred, z_target).item()
                errors.append(mse)

            mse_per_step_mean[s - 1] = np.mean(errors) if errors else 0.0
            mse_per_step_std[s - 1]  = np.std(errors)  if errors else 0.0

        return mse_per_step_mean, mse_per_step_std

    def _rollout_mse_gru(self, z, lengths):
        """
        Fallback rollout for GRU baseline using forward_rollout.
        Uses the encoded starting state z_0 as seed and rolls out autoregressively.
        """
        z_cpu = z.cpu()
        mse_per_step_mean = np.zeros(self.rollout_steps)
        mse_per_step_std  = np.zeros(self.rollout_steps)

        for s in range(1, self.rollout_steps + 1):
            errors = []
            for b, T in enumerate(lengths):
                if T <= s + 1:
                    continue
                # Use the GRU model's own rollout method for fairness
                seed = z_cpu[b:b+1, :1, :]   # (1, 1, latent_dim) — first frame
                preds = self.baseline_model.forward_rollout(
                    seed.to(self.device), steps=s + 1, latent_seed = True
                ).cpu()
                z_pred   = preds[0, s]                   # (latent_dim,)
                z_target = z_cpu[b, s]                   # (latent_dim,)
                errors.append(torch.nn.functional.mse_loss(z_pred, z_target).item())

            mse_per_step_mean[s - 1] = np.mean(errors) if errors else 0.0
            mse_per_step_std[s - 1]  = np.std(errors)  if errors else 0.0

        return mse_per_step_mean, mse_per_step_std

    # ── Metric: spectral radii ───────────────────────────────────────────────

    def _spectral_radii(self, K):
        """
        Computes spectral radius of the kinetic block (expected 1.0)
        and the phenotypic block.

        Key term — spectral radius ρ(A):
            The largest absolute eigenvalue of matrix A.
            ρ = 1.0  →  conservative (volume-preserving) dynamics
            ρ < 1.0  →  dissipative (contracting) dynamics
            ρ > 1.0  →  expansive (diverging) dynamics
        """
        K_np       = K.numpy()
        K_kin      = K_np[:self.sub_dim, :self.sub_dim]
        K_pheno    = K_np[self.sub_dim:, self.sub_dim:]
        eigvals    = np.linalg.eigvals

        rho_kin   = float(np.max(np.abs(eigvals(K_kin))))
        rho_pheno = float(np.max(np.abs(eigvals(K_pheno))))

        return rho_kin, rho_pheno

    # ── Metric: geometry retention ───────────────────────────────────────────

    def _geometry_retention(self, z, lengths, K):
        """
        Pairwise latent distance retention over the rollout horizon.

        At each step s, for a randomly sampled pair of sequences (i, j),
        compute the ratio of predicted pairwise distance to true pairwise
        distance at t=0:

            ratio_s = ||pred_i(s) - pred_j(s)|| / ||z_i(0) - z_j(0)||

        A ratio of 1.0 means the operator perfectly preserved pairwise geometry.
        Ratio → 0 means the operator collapsed both trajectories to the same point
        (the GRU attractor trap).

        Key term — isometry:
            A map f is an isometry if it preserves all pairwise distances.
            An orthogonal operator K is an isometry by construction:
            ||Kx - Ky|| = ||K(x-y)|| = ||x-y||  (because K preserves norms).

        Returns (geom_mean, geom_std) each shape (rollout_steps + 1,)
        Index 0 = t=0 (always 1.0 before normalization).
        """
        z_cpu  = z.cpu()
        N      = len(lengths)
        n_pairs = min(200, N * (N - 1) // 2)   # cap for speed

        # Sample random pairs
        rng   = np.random.default_rng(42)
        idx_i = rng.integers(0, N, size=n_pairs)
        idx_j = rng.integers(0, N, size=n_pairs)
        # Avoid self-pairs
        mask  = idx_i != idx_j
        idx_i, idx_j = idx_i[mask], idx_j[mask]

        geom_means = np.zeros(self.rollout_steps + 1)
        geom_stds  = np.zeros(self.rollout_steps + 1)

        if K is None:
            # GRU has no K — return zeros (handled in plot as N/A)
            return geom_means, geom_stds

        for s in range(self.rollout_steps + 1):
            if s == 0:
                K_pow = torch.eye(z_cpu.shape[-1])
            else:
                K_pow = torch.matrix_power(K, s)

            ratios = []
            for i, j in zip(idx_i, idx_j):
                T_i = min(lengths[i], lengths[j])
                if T_i < 1:
                    continue

                # Use first timestep as reference
                z_i0 = z_cpu[i, 0]   # (latent_dim,)
                z_j0 = z_cpu[j, 0]   # (latent_dim,)
                d0   = torch.norm(z_i0 - z_j0).item()
                if d0 < 1e-8:
                    continue

                # Predict both trajectories s steps forward
                z_i_pred = torch.matmul(z_i0.unsqueeze(0), K_pow.t()).squeeze()
                z_j_pred = torch.matmul(z_j0.unsqueeze(0), K_pow.t()).squeeze()
                d_pred   = torch.norm(z_i_pred - z_j_pred).item()

                ratios.append(d_pred / d0)

            geom_means[s] = np.mean(ratios) if ratios else 0.0
            geom_stds[s]  = np.std(ratios)  if ratios else 0.0

        return geom_means, geom_stds

    # ── Metric: collapse diagnostic ──────────────────────────────────────────

    def _relative_change(self, z, lengths):
        """
        Scale-invariant collapse indicator.

        relative_change = mean over all (b,t): ||z_{t+1} - z_t|| / ||z_t||

        > 0.05 → encoder is producing dynamic (non-collapsed) trajectories
        ≈ 0    → fixed-point collapse (all inputs map to the same vector)

        Key term — fixed-point collapse:
            The encoder maps all inputs to nearly the same latent vector.
            This trivially minimises the dynamics loss (K≈I satisfies it)
            but destroys all representational content.
        """
        z_cpu = z.cpu()
        all_ratios = []

        for b, T in enumerate(lengths):
            if T < 2:
                continue
            z_seq    = z_cpu[b, :T]                             # (T, latent_dim)
            diffs    = (z_seq[1:] - z_seq[:-1]).norm(dim=1)     # (T-1,)
            norms    = z_seq[:-1].norm(dim=1)                   # (T-1,)
            ratios   = (diffs / (norms + 1e-6)).tolist()
            all_ratios.extend(ratios)

        return float(np.mean(all_ratios)) if all_ratios else 0.0

    # ── Reporting ────────────────────────────────────────────────────────────

    def print_summary(self, results: EvalResults):
        """Prints a readable summary of all four metric groups."""
        sep = "=" * 70
        print(sep)
        print(f"EVALUATION SUMMARY — {results.meta.get('dataset','?')} "
              f"{results.meta.get('molecule','')}")
        print(sep)

        print("\n[1] ROLLOUT MSE")
        print(f"  {'Step':>4}  {'Koopman':>10}  {'Baseline':>10}")
        for s in range(len(results.koopman_mse_mean)):
            print(f"  {s+1:>4}  "
                  f"{results.koopman_mse_mean[s]:>10.4f}  "
                  f"{results.baseline_mse_mean[s]:>10.4f}")

        print("\n[2] SPECTRAL RADII")
        print(f"  rho(K_kin)  : {results.rho_kin:.6f}  "
              f"{'PASS (conservative)' if abs(results.rho_kin - 1.0) < 1e-3 else 'WARN'}")
        print(f"  rho(K_pheno): {results.rho_pheno:.4f}")

        final_koop = results.koopman_geom_mean[-1]
        final_base = results.baseline_geom_mean[-1]
        print("\n[3] GEOMETRY RETENTION @ final step")
        print(f"  Koopman : {final_koop:.4f}")
        print(f"  Baseline: {final_base:.4f}")

        print("\n[4] COLLAPSE DIAGNOSTIC")
        print(f"  Koopman  relative change: {results.relative_change_koopman:.4f}  "
              f"{'PASS' if results.relative_change_koopman > 0.05 else 'FAIL'}")
        print(f"  Baseline relative change: {results.relative_change_baseline:.4f}")
        print(sep)

    def plot(self, results: EvalResults, title: Optional[str] = None,
             save_path: Optional[str] = None):
        """
        Two-panel tradeoff plot:
          Top    — Rollout MSE vs prediction horizon
          Bottom — Normalized pairwise geometry retention vs horizon
        """
        KOOP_COLOR = "#2166ac"
        BASE_COLOR = "#d6604d"

        steps_err  = np.arange(1, self.rollout_steps + 1)
        steps_geom = np.arange(0, self.rollout_steps + 1)

        # Normalize geometry to t=0
        k0 = results.koopman_geom_mean[0] + 1e-8
        b0 = results.baseline_geom_mean[0] + 1e-8
        koop_geom_norm = results.koopman_geom_mean  / k0
        base_geom_norm = results.baseline_geom_mean / b0
        koop_gstd_norm = results.koopman_geom_std   / k0
        base_gstd_norm = results.baseline_geom_std  / b0

        dataset_label = (title or
                         f"{results.meta.get('dataset','?')} "
                         f"{results.meta.get('molecule','')}")

        fig = plt.figure(figsize=(13, 8))
        gs  = gridspec.GridSpec(2, 1, hspace=0.3)

        # ── Top: MSE ──────────────────────────────────────────────────────
        ax1 = fig.add_subplot(gs[0])
        ax1.plot(steps_err, results.koopman_mse_mean,
                 color=KOOP_COLOR, linewidth=2.5, marker='o', markersize=4,
                 label='Koopman (Orthogonal)')
        ax1.fill_between(steps_err,
                         results.koopman_mse_mean - results.koopman_mse_std,
                         results.koopman_mse_mean + results.koopman_mse_std,
                         color=KOOP_COLOR, alpha=0.15)
        ax1.plot(steps_err, results.baseline_mse_mean,
                 color=BASE_COLOR, linewidth=2.5, linestyle='--',
                 marker='x', markersize=5, label='GRU Baseline')
        ax1.fill_between(steps_err,
                         results.baseline_mse_mean - results.baseline_mse_std,
                         results.baseline_mse_mean + results.baseline_mse_std,
                         color=BASE_COLOR, alpha=0.15)
        ax1.set_ylabel("Rollout MSE", fontsize=12)
        ax1.set_title(
            f"Predictive Accuracy vs. Geometric Stability — {dataset_label}",
            fontsize=13, fontweight='bold', pad=10)
        ax1.legend(fontsize=11, loc='upper left')
        ax1.grid(True, linestyle=':', alpha=0.5)
        ax1.set_xticklabels([])

        # ── Bottom: geometry retention ────────────────────────────────────
        ax2 = fig.add_subplot(gs[1], sharex=ax1)
        ax2.plot(steps_geom, koop_geom_norm,
                 color=KOOP_COLOR, linewidth=2.5, marker='o', markersize=4)
        ax2.fill_between(steps_geom,
                         koop_geom_norm - koop_gstd_norm,
                         koop_geom_norm + koop_gstd_norm,
                         color=KOOP_COLOR, alpha=0.15)
        ax2.plot(steps_geom, base_geom_norm,
                 color=BASE_COLOR, linewidth=2.5, linestyle='--', marker='x',
                 markersize=5)
        ax2.fill_between(steps_geom,
                         base_geom_norm - base_gstd_norm,
                         base_geom_norm + base_gstd_norm,
                         color=BASE_COLOR, alpha=0.15)
        ax2.axhline(1.0, color='gray', linewidth=1.0, linestyle=':',
                    alpha=0.7, label='Perfect retention (ratio = 1.0)')
        ax2.set_xlabel("Prediction Horizon (steps)", fontsize=12)
        ax2.set_ylabel("Pairwise Distance Ratio\n(normalized to t=0)", fontsize=12)
        ax2.legend(fontsize=10, loc='lower left')
        ax2.grid(True, linestyle=':', alpha=0.5)

        # Adjust subplot parameters for a tight layout
        fig.subplots_adjust(left=0.08, right=0.95, top=0.9, bottom=0.1, hspace=0.3)

        if save_path:
            plt.savefig(save_path, dpi=150, bbox_inches='tight')
        plt.show()

        # Print key numbers
        print("\nTRADEOFF SUMMARY")
        print(f"{'Metric':<40} {'Koopman':>10} {'Baseline':>10}")
        print("-" * 62)
        print(f"{'MSE @ step 1 (short horizon)':<40} "
              f"{results.koopman_mse_mean[0]:>10.4f} "
              f"{results.baseline_mse_mean[0]:>10.4f}")
        print(f"{'MSE @ final step':<40} "
              f"{results.koopman_mse_mean[-1]:>10.4f} "
              f"{results.baseline_mse_mean[-1]:>10.4f}")
        print(f"{'Geometry retention @ final step':<40} "
              f"{koop_geom_norm[-1]:>10.4f} "
              f"{base_geom_norm[-1]:>10.4f}")

In [ ]:
# ── Cell F: BlockDiagonalKoopmanNet with parameterized hidden_dim ─────────
# Changes from previous version:
#   - hidden_dim added as constructor argument (default 64 — backward compatible)
#   - Both encoder_kin and encoder_pheno use hidden_dim
#   - temporal_conv out_channels scales with hidden_dim // 2
#   - All other logic unchanged

class BlockDiagonalKoopmanNet(nn.Module):
    """
    Partitioned 32D Koopman network with two structurally isolated subspaces.

    Parameters
    ──────────
    input_dim  : int
                 Raw feature dimension (number of pairwise distances).
                 Encoder receives input_dim * 2 (position + velocity).
    latent_dim : int
                 Must be even. Split 50/50 into kinetic and phenotypic.
                 Default 32.
    hidden_dim : int
                 Width of the hidden layer in both encoders.
                 Rule of thumb: set to max(64, input_dim * 2 // 4).
                 Default 64 — matches all previously saved ethanol checkpoints.
                 For aspirin (input_dim=210): use 128 or 256.
    """

    def __init__(self, input_dim, latent_dim=32, hidden_dim=64):
        super().__init__()

        assert latent_dim % 2 == 0, "latent_dim must be even"
        self.sub_dim    = latent_dim // 2
        self.latent_dim = latent_dim
        self.hidden_dim = hidden_dim

        phase_space_dim = input_dim * 2   # position || velocity

        # ── Encoder 1: Kinetic Core ───────────────────────────────────
        self.encoder_kin = nn.Sequential(
            nn.Linear(phase_space_dim, hidden_dim),
            nn.GELU(),
            nn.LayerNorm(hidden_dim),
            nn.Linear(hidden_dim, self.sub_dim)
        )

        # ── Encoder 2: Phenotypic Subspace ────────────────────────────
        self.encoder_pheno = nn.Sequential(
            nn.Linear(phase_space_dim, hidden_dim),
            nn.GELU(),
            nn.LayerNorm(hidden_dim),
            nn.Linear(hidden_dim, self.sub_dim)
        )

        # ── Kinetic Operator — skew-symmetric exponential map ─────────
        self.A_raw = nn.Parameter(
            torch.randn(self.sub_dim, self.sub_dim) * 0.02
        )

        # ── Phenotypic Operator — unconstrained ───────────────────────
        self.K_pheno_raw = nn.Parameter(
            torch.randn(self.sub_dim, self.sub_dim) * 0.02
        )

        # ── Classifier: 1D-TCN over phenotypic trajectory ─────────────
        # out_channels scales with hidden_dim so the classifier capacity
        # grows proportionally with the encoder capacity.
        #
        # Key term — Conv1d kernel_size=3, padding=1:
        #            Each output timestep sees the current timestep plus
        #            one neighbour on each side. padding=1 ensures the
        #            output length equals the input length (same padding).
        tcn_out = max(32, hidden_dim // 2)   # e.g. 64 for hidden_dim=128

        self.temporal_conv = nn.Sequential(
            nn.Conv1d(
                in_channels  = self.sub_dim,
                out_channels = tcn_out,
                kernel_size  = 3,
                padding      = 1
            ),
            nn.GELU(),
            nn.BatchNorm1d(tcn_out),
            nn.AdaptiveAvgPool1d(1)           # (B, tcn_out, T) → (B, tcn_out, 1)
        )

        self.classifier = nn.Sequential(
            nn.Linear(tcn_out, 16),
            nn.GELU(),
            nn.Linear(16, 2)
        )

    @property
    def K(self):
        """Assembles full 32×32 block-diagonal operator."""
        device = self.A_raw.device
        A      = self.A_raw - self.A_raw.t()     # skew-symmetrize
        K_kin  = torch.matrix_exp(A)             # guaranteed orthogonal

        zeros  = torch.zeros(self.sub_dim, self.sub_dim, device=device)
        K_top  = torch.cat([K_kin,  zeros          ], dim=1)
        K_bot  = torch.cat([zeros,  self.K_pheno_raw], dim=1)

        return torch.cat([K_top, K_bot], dim=0)  # (32, 32)

    def forward(self, x, lengths):
        """
        Parameters
        ──────────
        x       : Tensor (B, max_T, phase_space_dim)
        lengths : list[int]

        Returns
        ───────
        z_kin      : Tensor (B, max_T, sub_dim)
        z_pheno    : Tensor (B, max_T, sub_dim)
        cls_logits : Tensor (B, 2)
        """
        B, max_T, D = x.shape
        x_flat = x.contiguous().view(B * max_T, D)

        z_kin_flat   = self.encoder_kin(x_flat)
        z_pheno_flat = self.encoder_pheno(x_flat)

        z_kin   = z_kin_flat.view(B, max_T, self.sub_dim)
        z_pheno = z_pheno_flat.view(B, max_T, self.sub_dim)

        # Classification path — phenotypic subspace only
        # transpose: (B, T, sub_dim) → (B, sub_dim, T) for Conv1d
        z_pheno_t        = z_pheno.transpose(1, 2)
        z_pheno_isolated = z_pheno_t.detach()
        temporal_out     = self.temporal_conv(z_pheno_isolated)
        temporal_out     = temporal_out.squeeze(-1)       # (B, tcn_out)
        cls_logits       = self.classifier(temporal_out)  # (B, 2)

        return z_kin, z_pheno, cls_logits


print("BlockDiagonalKoopmanNet (parameterized hidden_dim) defined.")

# ── Cell G: AnchoredBlockDiagonalKoopmanNet with hidden_dim passthrough ───
# Only change: hidden_dim is passed through to super().__init__()
# BatchNorm layers adapt automatically to sub_dim (unchanged)

class AnchoredBlockDiagonalKoopmanNet(BlockDiagonalKoopmanNet):
    """
    Adds BatchNorm variance anchoring to both encoder outputs.

    Key term — BatchNorm1d(affine=False):
               Normalizes activations to zero mean and unit variance
               across the batch. affine=False means no learnable scale
               or shift — pure normalization only.
               This prevents the microscopic variance collapse that
               caused R² denominator instability in earlier versions.

    Parameters
    ──────────
    input_dim  : int   — passed through to BlockDiagonalKoopmanNet
    latent_dim : int   — default 32
    hidden_dim : int   — default 64, increase for larger molecules
    """

    def __init__(self, input_dim, latent_dim=32, hidden_dim=64):
        super().__init__(input_dim, latent_dim, hidden_dim)

        # Variance anchoring layers — one per encoder output
        # sub_dim is inherited from parent: latent_dim // 2
        self.kin_norm   = nn.BatchNorm1d(self.sub_dim, affine=False)
        self.pheno_norm = nn.BatchNorm1d(self.sub_dim, affine=False)

    def forward(self, x, lengths):
        B, max_T, D = x.shape
        x_flat = x.contiguous().view(B * max_T, D)

        z_kin_flat   = self.encoder_kin(x_flat)
        z_pheno_flat = self.encoder_pheno(x_flat)

        # Anchor variance — applied flat before reshaping back to (B, T, D)
        # BatchNorm1d expects (N, C): here N = B*max_T, C = sub_dim
        z_kin_flat   = self.kin_norm(z_kin_flat)
        z_pheno_flat = self.pheno_norm(z_pheno_flat)

        z_kin   = z_kin_flat.view(B, max_T, self.sub_dim)
        z_pheno = z_pheno_flat.view(B, max_T, self.sub_dim)

        z_pheno_t        = z_pheno.transpose(1, 2)
        z_pheno_isolated = z_pheno_t.detach()
        temporal_out     = self.temporal_conv(z_pheno_isolated)
        temporal_out     = temporal_out.squeeze(-1)
        cls_logits       = self.classifier(temporal_out)

        return z_kin, z_pheno, cls_logits


print("AnchoredBlockDiagonalKoopmanNet (hidden_dim passthrough) defined.")

In [ ]:
# ── Cell H: BlackBoxGraphDynamicsNet with parameterized hidden_dim ─────────

class BlackBoxGraphDynamicsNet(nn.Module):
    """
    GRU baseline with parameterized encoder capacity.

    Parameters
    ──────────
    input_dim  : int   — phase_space_dim (input_dim_raw * 2 from adapter)
    latent_dim : int   — default 32
    hidden_dim : int   — default 64, increase for larger molecules
    """

    def __init__(self, input_dim, latent_dim=32, hidden_dim=64):
        super().__init__()

        self.latent_dim = latent_dim
        self.hidden_dim = hidden_dim

        # Key term — input_dim here is already phase_space_dim (doubled),
        #            because the GRU receives the full velocity-augmented
        #            feature vector, not the raw distance vector.
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.GELU(),
            nn.LayerNorm(hidden_dim),
            nn.Linear(hidden_dim, latent_dim)
        )

        self.rnn_transition = nn.GRUCell(latent_dim, latent_dim)

        self.classifier = nn.Sequential(
            nn.Linear(latent_dim, 16),
            nn.GELU(),
            nn.Linear(16, 2)
        )

    def forward_rollout(self, x, steps=30, latent_seed=False):
        """
        Autonomous rollout from raw input or pre-encoded latent seed.

        Parameters
        ──────────
        x           : Tensor (B, T, D) raw, or (B, 1, latent_dim) if latent_seed=True
        steps       : int
        latent_seed : bool
                      True when called from the evaluator — x[:,0] is already
                      a latent vector, skip encoding.
                      False for standard use — x[:,0] is raw phase-space features.
        """
        if latent_seed:
            z = x[:, 0]                  # (B, latent_dim) — already encoded
        else:
            z = self.encoder(x[:, 0])    # (B, latent_dim) — encode first

        rollout_preds = [z]
        for _ in range(1, steps):
            gru_input = torch.zeros_like(z)
            z = self.rnn_transition(gru_input, z)
            rollout_preds.append(z)

        return torch.stack(rollout_preds, dim=1)   # (B, steps, latent_dim)

    def forward(self, x, lengths):
        B, T, D = x.shape
        x_flat  = x.contiguous().view(B * T, D)
        z_flat  = self.encoder(x_flat)
        z_seq   = z_flat.view(B, T, self.latent_dim)

        # Pool last valid timestep per sequence for classification
        pooled = torch.stack([z_seq[b, lengths[b] - 1] for b in range(B)])
        logits = self.classifier(pooled)

        return z_seq, logits


print("BlackBoxGraphDynamicsNet (parameterized hidden_dim) defined.")

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class RegularizedGRUBaselineNet(BlackBoxGraphDynamicsNet):
    def __init__(self, input_dim, latent_dim=32, hidden_dim=64, beta=1.0):
        super().__init__(input_dim, latent_dim, hidden_dim)
        self.beta = beta

    def compute_loss(self, outputs, targets, lengths, epoch):
        z_seq, cls_logits = outputs

        # 1. Dynamics loss (one-step GRU prediction)
        z_t      = z_seq[:, :-1]
        z_target = z_seq[:, 1:]

        B, Tm1, D   = z_t.shape
        z_t_flat    = z_t.reshape(B * Tm1, D)
        z_tgt_flat  = z_target.reshape(B * Tm1, D)

        gru_input   = torch.zeros_like(z_t_flat)
        z_pred_flat = self.rnn_transition(gru_input, z_t_flat)

        l_dyn = F.mse_loss(z_pred_flat, z_tgt_flat)

        # 2. Classification loss
        criterion = nn.CrossEntropyLoss()
        l_cls     = criterion(cls_logits, targets)

        # 3. Variance hinge (prevents encoder collapse)
        latent_var = z_seq.var(dim=1).mean()
        l_var      = torch.relu(0.1 - latent_var)

        # 4. Latent Norm Preservation loss
        # Penalize difference in L2 norms between prediction and current step
        norm_t = torch.norm(z_t_flat, p=2, dim=-1)
        norm_pred = torch.norm(z_pred_flat, p=2, dim=-1)
        l_norm = F.mse_loss(norm_pred, norm_t)

        total_loss = l_dyn + l_cls + 0.5 * l_var + self.beta * l_norm

        log = {
            'loss' : total_loss.item(),
            'l_dyn': l_dyn.item(),
            'l_cls': l_cls.item(),
            'l_var': l_var.item(),
            'l_norm': l_norm.item(),
        }

        return total_loss, log

    def post_step_hook(self):
        pass

print("RegularizedGRUBaselineNet defined successfully.")


In [ ]:
# import torch
# import os

# koopman_model = AnchoredBlockDiagonalKoopmanNet(input_dim=36, latent_dim=32)

# # 1. Load the entire checkpoint dictionary
# # (Using weights_only=False since we know it contains custom training metadata metrics)
# koopman_checkpoint = torch.load(
#     os.path.join(abhilash437_anchored_block_diagonal_koopman_model_pytorch_default_1_path, 'anchored_block_diagonal_koopman_model_1.pt'),
#     weights_only=False
# )

# # 2. Extract ONLY the model weights from the dictionary
# koopman_state_dict = koopman_checkpoint["model_state_dict"]

# # 3. Inject the weights into your model instance
# koopman_model.load_state_dict(koopman_state_dict)
# koopman_model

In [ ]:
# import torch
# import os

# baseline_model = BlackBoxGraphDynamicsNet(input_dim=72, latent_dim=16)

# # 1. Load the entire checkpoint dictionary
# # (Using weights_only=False since we know it contains custom training metadata metrics)
# baseline_checkpoint = torch.load(
#     os.path.join(abhilash437_gru_baseline_pytorch_default_1_path, 'gru_baseline_epoch_100.pt'),
#     weights_only=False
# )

# # 2. Extract ONLY the model weights from the dictionary
# baseline_state_dict = baseline_checkpoint["model_state_dict"]

# # 3. Inject the weights into your model instance
# baseline_model.load_state_dict(baseline_state_dict)
# baseline_model

In [ ]:
# evaluator = KoopmanEvaluator(koopman_model, baseline_model)
# results = evaluator.run(test_split)
# evaluator.print_summary(results)
# evaluator.plot(results)

In [ ]:
# ── Cell A: Koopman compute_loss and post_step_hook ──────────────────────
# Adds two methods to AnchoredBlockDiagonalKoopmanNet without touching
# the existing class definition.
#
# Key term — monkey patching: attaching methods to a class after it has
#            been defined. We use this to keep the original class cells
#            unchanged while extending behaviour for the Trainer.

import types   # needed for types.MethodType — binds a function to an instance

CLASS_WEIGHTS_KOOPMAN = torch.tensor([1.0, 0.1181])

def project_phenotypic_to_unitary(model):
    """
    Extracts the raw phenotypic matrix, projects it to the closest
    orthogonal manifold via SVD, and writes it back to hold rho = 1.0.
    """
    with torch.no_grad():
        # Access your raw phenotypic parameter block from the model
        # Note: adjust parameter name if your class stores it differently (e.g., self.K_pheno_raw)
        K_raw = model.K_pheno_raw

        # Compute Singular Value Decomposition
        U, S, Vh = torch.linalg.svd(K_raw, full_matrices=False)

        # Project back to a perfect unitary matrix (forcing singular values to 1.0)
        K_orthogonal = torch.matmul(U, Vh)

        # Copy back in-place to preserve parameter tracking
        model.K_pheno_raw.copy_(K_orthogonal)

def get_lambda_schedule(epoch):
    """
    Three-phase warmup schedule for dynamics/classification loss balance.

    Phase 1 (epochs  1–15) : Classification dominates.
                             Encoder_pheno learns a good representation
                             before dynamics loss starts competing.
    Phase 2 (epochs 16–40) : Balanced. Both losses contribute equally.
    Phase 3 (epochs 41+)   : Dynamics dominates.
                             Kinetic core is pushed toward linear structure.

    Returns
    ───────
    lambda_dyn, lambda_cls : float, float
    """
    lambda_dyn = 1.0
    lambda_cls = 5.0 if epoch < 30 else 2.5
    return lambda_dyn, lambda_cls

def compute_partitioned_loss(model, z_kin, z_pheno, cls_logits,
                              targets, lengths,
                              lambda_dyn=1.0, lambda_cls=1.0,
                              lambda_collapse=2.0,
                              class_weights=None):
    """
    Three-term partitioned loss.

    l_dyn      : linear predictability of each subspace via its own operator
    l_cls      : cross-entropy on phenotypic classifier output
    l_collapse : anti-collapse guard on kinetic encoder
                 Penalizes low temporal variance in z_kin per sequence.

    Parameters
    ──────────
    model          : BlockDiagonalKoopmanNet
    z_kin          : Tensor (B, max_T, 16)
    z_pheno        : Tensor (B, max_T, 16)
    cls_logits     : Tensor (B, 2)
    targets        : Tensor (B,)
    lengths        : list[int]
    lambda_dyn     : float
    lambda_cls     : float
    lambda_collapse: float — weight on anti-collapse penalty.
                     Higher values enforce more spread in z_kin.
                     Start at 2.0 — same order as lambda_dyn in Phase 1.
    class_weights  : Tensor (2,) or None
    """

    # ── Dynamics loss ─────────────────────────────────────────────────────
    kin_true,   kin_pred   = [], []
    pheno_true, pheno_pred = [], []

    K_kin   = model.K[:model.sub_dim, :model.sub_dim]
    K_pheno = model.K[model.sub_dim:, model.sub_dim:]

    # ── Collapse guard ────────────────────────────────────────────────────
    # Measures mean relative step size across consecutive timesteps.
    #
    # relative_change_t = ||z_{t+1} - z_t|| / (||z_t|| + eps)
    #
    # Key term — relative change: normalises the step size by the
    #            current state magnitude. Scale-invariant by construction —
    #            the encoder cannot escape this penalty by shrinking outputs.
    #
    # Key term — hinge penalty: torch.relu(threshold - x) returns 0
    #            when x >= threshold (healthy), (threshold - x) when
    #            x < threshold (collapsing). Only penalises collapse,
    #            never rewards excess movement.
    #
    # threshold = 0.05: consecutive states should differ by at least
    #             5% of their magnitude. Well below any meaningful
    #             dynamics signal but well above fixed-point collapse.

    collapse_penalties = []
    eps = 1e-6

    for b in range(len(lengths)):
        T = lengths[b]

        if T < 2:
            continue

        z_seq = z_kin[b, :T]                           # (T, 16)

        # Step vectors between consecutive states
        diffs  = z_seq[1:] - z_seq[:-1]                # (T-1, 16)

        # L2 norm of each step and each state
        step_norms  = diffs.norm(dim=1)                 # (T-1,)
        state_norms = z_seq[:-1].norm(dim=1)            # (T-1,)

        # Relative change per timestep
        relative_change = step_norms / (state_norms + eps)   # (T-1,)

        # Mean relative change across the sequence
        mean_rel_change = relative_change.mean()

        # Penalise if below threshold
        penalty = torch.relu(0.05 - mean_rel_change)
        collapse_penalties.append(penalty)

    if len(collapse_penalties) > 0:
        l_collapse = torch.stack(collapse_penalties).mean()
    else:
        l_collapse = z_kin.new_tensor(0.0)

    # ── Dynamics loss (unchanged from Step A) ────────────────────────────
    for b in range(len(lengths)):
        T = lengths[b]
        kin_pred.append(
            torch.matmul(z_kin[b, :T-1], K_kin.t())
        )
        kin_true.append(z_kin[b, 1:T])

        pheno_pred.append(
            torch.matmul(z_pheno[b, :T-1], K_pheno.t())
        )
        pheno_true.append(z_pheno[b, 1:T])

    kin_pred   = torch.cat(kin_pred,   dim=0)
    kin_true   = torch.cat(kin_true,   dim=0)
    pheno_pred = torch.cat(pheno_pred, dim=0)
    pheno_true = torch.cat(pheno_true, dim=0)

    l_dyn = (nn.functional.mse_loss(kin_pred,   kin_true) +
             nn.functional.mse_loss(pheno_pred, pheno_true))

    # ── Classification loss (unchanged from Step A) ───────────────────────
    criterion_cls = nn.CrossEntropyLoss(weight=class_weights.to(targets.device))
    l_cls = criterion_cls(cls_logits, targets)

    total_loss = (lambda_dyn      * l_dyn +
                  lambda_cls      * l_cls +
                  lambda_collapse * l_collapse)

    return total_loss, l_dyn.detach(), l_cls.detach(), l_collapse.detach()

def koopman_compute_loss(self, outputs, targets, lengths, epoch):
    """
    Computes the full Koopman training loss for one batch.

    Parameters
    ──────────
    outputs : tuple (z_kin, z_pheno, cls_logits)
              Directly from model.forward(x, lengths).
    targets : Tensor (B,)   integer class labels
    lengths : list[int]     true unpadded sequence lengths
    epoch   : int           current epoch, used by lambda schedule

    Returns
    ───────
    total_loss : Tensor     scalar, call .backward() on this
    log        : dict       detached floats for printing/checkpointing
                            keys: 'loss', 'l_dyn', 'l_cls', 'l_collapse',
                                  'l_ortho', 'l_var'
    """
    z_kin, z_pheno, cls_logits = outputs

    # ── Scheduled loss weights ────────────────────────────────────────
    lambda_dyn, lambda_cls = get_lambda_schedule(epoch)

    # ── Base partitioned loss ─────────────────────────────────────────
    total_loss, l_dyn_val, l_cls_val, l_collapse_val = compute_partitioned_loss(
        model           = self,
        z_kin           = z_kin,
        z_pheno         = z_pheno,
        cls_logits      = cls_logits,
        targets         = targets,
        lengths         = lengths,
        lambda_dyn      = lambda_dyn,
        lambda_cls      = lambda_cls,
        lambda_collapse = 2.0,
        class_weights   = CLASS_WEIGHTS_KOOPMAN
    )

    # ── Cross-covariance orthogonality penalty ────────────────────────
    # Penalises shared variance between kinetic and phenotypic subspaces.
    #
    # Key term — Frobenius norm: for a matrix M, ||M||_F = sqrt(sum of
    #            all squared entries). Used here to penalise any non-zero
    #            entry in the cross-covariance matrix, not just the diagonal.
    #
    # Why center first: raw covariance mixes variance (diagonal) with
    # covariance (off-diagonal). Centering removes the mean so the
    # Frobenius norm measures pure cross-subspace correlation.
    z_kin_c   = z_kin   - z_kin.mean(dim=0, keepdim=True)
    z_pheno_c = z_pheno - z_pheno.mean(dim=0, keepdim=True)

    # Reshape (B, T, 16) → (B*T, 16) before matmul
    # Dividing by B*T gives the mean cross-covariance per sample
    BT = z_kin.shape[0] * z_kin.shape[1]
    cross_cov = torch.matmul(
        z_kin_c.view(-1, 16).t(),
        z_pheno_c.view(-1, 16)
    ) / BT                                     # (16, 16)

    l_ortho = torch.norm(cross_cov, p='fro')

    # ── Variance hinge ────────────────────────────────────────────────
    # Prevents the kinetic encoder from collapsing to near-constant
    # outputs even when the relative-change guard is active.
    #
    # Key term — variance hinge: torch.relu(threshold - var) returns 0
    #            when variance is healthy, and a positive penalty when
    #            variance drops below the threshold.
    #
    # var(dim=1): variance across timesteps T, one value per
    #             (batch element, latent dimension) pair.
    kin_var     = z_kin.var(dim=1)                         # (B, 16)
    l_var_hinge = torch.relu(0.1 - kin_var).mean()

    # ── Combine ───────────────────────────────────────────────────────
    total_loss = total_loss + 5.0 * l_ortho + 1.0 * l_var_hinge

    log = {
        'loss'      : total_loss.item(),
        'l_dyn'     : l_dyn_val.item(),
        'l_cls'     : l_cls_val.item(),
        'l_collapse': l_collapse_val.item(),
        'l_ortho'   : l_ortho.item(),
        'l_var'     : l_var_hinge.item(),
    }

    return total_loss, log


def koopman_post_step_hook(self):
    """
    Projects K_pheno_raw back to the nearest orthogonal matrix after
    each gradient step.

    Called after optimizer.step(). Keeps rho(K_pheno) = 1.0 without
    adding a gradient penalty — the projection is applied in-place with
    no_grad so it does not affect the gradient graph.
    """
    project_phenotypic_to_unitary(self)


# Bind methods to the class (affects all instances, including saved ones)
AnchoredBlockDiagonalKoopmanNet.compute_loss    = koopman_compute_loss
AnchoredBlockDiagonalKoopmanNet.post_step_hook  = koopman_post_step_hook

print("Koopman compute_loss and post_step_hook attached.")

In [ ]:
# ── Cell B: GRU compute_loss and post_step_hook ───────────────────────────

def gru_compute_loss(self, outputs, targets, lengths, epoch):
    """
    Computes the GRU baseline training loss for one batch.

    Parameters
    ──────────
    outputs : tuple (z_seq, cls_logits)
              Directly from model.forward(x, lengths).
    targets : Tensor (B,)
    lengths : list[int]
    epoch   : int           (unused for GRU — fixed weights)

    Returns
    ───────
    total_loss : Tensor
    log        : dict   keys: 'loss', 'l_dyn', 'l_cls', 'l_var'
    """
    z_seq, cls_logits = outputs

    # ── Dynamics loss ─────────────────────────────────────────────────
    # One-step prediction: GRUCell(zeros, z_t) should predict z_{t+1}.
    # Flatten time dimension to process all steps in one GRUCell call.
    #
    # Key term — reshape vs view: both reshape tensors, but view() requires
    #            contiguous memory. reshape() handles non-contiguous tensors
    #            safely by copying if needed.
    z_t      = z_seq[:, :-1]          # (B, T-1, D) — current states
    z_target = z_seq[:, 1:]           # (B, T-1, D) — next states

    B, Tm1, D   = z_t.shape
    z_t_flat    = z_t.reshape(B * Tm1, D)
    z_tgt_flat  = z_target.reshape(B * Tm1, D)

    # Autonomous step: no external input, hidden state carries everything
    gru_input   = torch.zeros_like(z_t_flat)
    z_pred_flat = self.rnn_transition(gru_input, z_t_flat)

    l_dyn = F.mse_loss(z_pred_flat, z_tgt_flat)

    # ── Classification loss ───────────────────────────────────────────
    criterion = nn.CrossEntropyLoss()
    l_cls     = criterion(cls_logits, targets)

    # ── Variance hinge ────────────────────────────────────────────────
    # Same guard as Koopman — prevents encoder collapse.
    # var(dim=1): variance across time axis T.
    latent_var = z_seq.var(dim=1).mean()
    l_var      = torch.relu(0.1 - latent_var)

    total_loss = l_dyn + l_cls + 0.5 * l_var

    log = {
        'loss' : total_loss.item(),
        'l_dyn': l_dyn.item(),
        'l_cls': l_cls.item(),
        'l_var': l_var.item(),
    }

    return total_loss, log


def gru_post_step_hook(self):
    """No-op for GRU — no post-step projection needed."""
    pass


BlackBoxGraphDynamicsNet.compute_loss   = gru_compute_loss
BlackBoxGraphDynamicsNet.post_step_hook = gru_post_step_hook

print("GRU compute_loss and post_step_hook attached.")

In [ ]:
# ── Cell C: Shared validation utilities ──────────────────────────────────
# These functions are model-agnostic — they only need encoded latent
# trajectories and predicted logits, regardless of which model produced them.

def compute_val_r2(model, z_kin, lengths):
    """
    Pearson R² on the kinetic subspace — measures how linearly predictable
    the latent trajectory is under the model's own operator K.

    For GRU: uses the full z_seq (all latent dims) as z_kin, and the
    GRU transition as the operator. But since GRU has no explicit K matrix,
    we compute K empirically via least squares on the validation trajectories.

    Key term — Pearson R²: computed per latent dimension d as:
        r_d = cov(pred_d, true_d) / (std(pred_d) * std(true_d))
        R²_d = r_d²
    Then averaged across all dimensions. This is scale-invariant — it
    measures structural alignment (shape of the trajectory) rather than
    absolute coordinate accuracy. A model that predicts the right direction
    but wrong magnitude still gets a good score.

    Why not standard R²: standard R² = 1 - SS_res/SS_tot penalises scale
    mismatches heavily. If BatchNorm squeezes activations, standard R²
    collapses even when the encoder is healthy. Pearson R² avoids this.

    Parameters
    ──────────
    model   : nn.Module — must expose model.K (Koopman) or rnn_transition (GRU)
    z_kin   : Tensor (B, max_T, D) — kinetic latent trajectory
    lengths : list[int]

    Returns
    ───────
    val_r2 : float
    """
    # ── Build predicted and true next-step pairs ──────────────────────
    preds_list, trues_list = [], []

    if hasattr(model, 'K'):
        # Koopman path — use the block operator directly
        K_kin = model.K[:model.sub_dim, :model.sub_dim]
        K_t   = K_kin.t()

        for b in range(len(lengths)):
            T = lengths[b]
            if T <= 1:
                continue
            preds_list.append(torch.matmul(z_kin[b, :T-1], K_t))
            trues_list.append(z_kin[b, 1:T])

    else:
        # GRU path — operator is the GRUCell with zero input
        # Apply one autonomous step to each valid z_t to get predicted z_{t+1}
        #
        # Key term — autonomous step: GRUCell(zeros, z_t) propagates the
        #            hidden state forward with no external driving signal.
        #            This is exactly how forward_rollout works at evaluation time.
        for b in range(len(lengths)):
            T = lengths[b]
            if T <= 1:
                continue
            z_t     = z_kin[b, :T-1]                        # (T-1, D)
            gru_inp = torch.zeros_like(z_t)
            z_pred  = model.rnn_transition(gru_inp, z_t)    # (T-1, D)
            preds_list.append(z_pred)
            trues_list.append(z_kin[b, 1:T])

    if len(preds_list) == 0:
        return 0.0

    preds_np = torch.cat(preds_list).detach().cpu().numpy()   # (N_pairs, D)
    trues_np = torch.cat(trues_list).detach().cpu().numpy()

    # ── Pearson R² per dimension, then average ────────────────────────
    correlations = []
    for d in range(trues_np.shape[1]):
        p = preds_np[:, d]
        t = trues_np[:, d]

        std_p = p.std()
        std_t = t.std()

        if std_p > 1e-6 and std_t > 1e-6:
            cov = np.mean((p - p.mean()) * (t - t.mean()))
            r   = cov / (std_p * std_t)
            correlations.append(r ** 2)

    return float(np.mean(correlations)) if correlations else 0.0


def compute_val_auc(cls_logits, targets):
    """
    ROC-AUC on the classifier output.

    Key term — softmax: converts raw logits (unbounded real numbers) into
               a probability distribution that sums to 1. We take [:, 1]
               to get the probability of the positive class.

    Returns 0.0 if only one class is present in the batch (roc_auc_score
    raises ValueError in that case — caught here to keep training stable).

    Parameters
    ──────────
    cls_logits : Tensor (B, 2)
    targets    : Tensor (B,)

    Returns
    ───────
    auc : float
    """
    probs  = torch.softmax(cls_logits, dim=-1)[:, 1].detach().cpu().numpy()
    labels = targets.detach().cpu().numpy()

    try:
        return float(roc_auc_score(labels, probs))
    except ValueError:
        return 0.0


print("Validation utilities defined: compute_val_r2, compute_val_auc")

In [ ]:
# ── Cell D: Trainer ───────────────────────────────────────────────────────
#
# Design contract:
#   - Accepts any model that implements compute_loss() and post_step_hook()
#   - Accepts any DatasetSplit from the adapter framework
#   - Saves best checkpoint by composite score: 0.5*max(R²,0) + 0.5*AUC
#   - Mini-batch training for both models (batch_size configurable)
#   - Validation uses stratified balanced sampling (same as original Koopman loop)

import os
import numpy as np
import torch
import torch.nn as nn
from sklearn.metrics import roc_auc_score

class Trainer:
    """
    Dataset-agnostic trainer for any model implementing compute_loss()
    and post_step_hook().

    Parameters
    ──────────
    model          : nn.Module
                     Must implement:
                       compute_loss(outputs, targets, lengths, epoch)
                         → (total_loss, log_dict)
                       post_step_hook()
                         → None  (no-op for GRU, SVD projection for Koopman)
                     Must implement standard forward(x, lengths)
                       → outputs tuple

    optimizer      : torch.optim.Optimizer
                     Constructed outside Trainer — keeps learning rate
                     scheduling decisions in the caller's hands.

    checkpoint_dir : str
                     Directory where best checkpoint is saved.

    checkpoint_name: str
                     Filename for the checkpoint (without path).

    epochs         : int    (default 100)
    batch_size     : int    (default 16)
    device         : str    (default 'cpu')
    log_every      : int    epochs between printed log lines (default 10)
    """

    def __init__(
        self,
        model,
        optimizer,
        checkpoint_dir,
        checkpoint_name,
        epochs         = 100,
        batch_size     = 16,
        device         = 'cpu',
        log_every      = 10,
    ):
        self.model           = model.to(device)
        self.optimizer       = optimizer
        self.checkpoint_dir  = checkpoint_dir
        self.checkpoint_name = checkpoint_name
        self.epochs          = epochs
        self.batch_size      = batch_size
        self.device          = device
        self.log_every       = log_every

        os.makedirs(checkpoint_dir, exist_ok=True)

        self._best_score     = -float('inf')
        self._best_meta      = {}

    # ── Public entry point ────────────────────────────────────────────────

    def fit(self, train_split, val_split):
        """
        Full training loop.

        Parameters
        ──────────
        train_split : DatasetSplit   produced by any DynamicsDatasetAdapter
        val_split   : DatasetSplit

        Returns
        ───────
        best_meta : dict
            Keys: 'epoch', 'val_r2', 'val_auc', 'composite', 'checkpoint_path'
        """
        # Convert DatasetSplit arrays to tensors once — avoids repeated
        # conversion inside the epoch loop.
        #
        # Key term — torch.from_numpy: creates a tensor that shares memory
        #            with the numpy array (zero-copy). We call .float() to
        #            ensure float32 regardless of the array's dtype.
        X_train = torch.from_numpy(train_split.X).float()
        y_train = torch.from_numpy(train_split.y).long()
        X_val   = torch.from_numpy(val_split.X).float()
        y_val   = torch.from_numpy(val_split.y).long()

        N_train  = len(X_train)
        header   = self._make_header()
        print(header)
        print("=" * len(header))

        for epoch in range(1, self.epochs + 1):

            # ── Training ──────────────────────────────────────────────
            train_log = self._train_epoch(
                X_train, y_train, train_split.lengths, epoch, N_train
            )

            # ── Validation ────────────────────────────────────────────
            val_r2, val_auc = self._validate(
                X_val, y_val, val_split.lengths
            )

            # ── Composite score and checkpoint ────────────────────────
            composite = 0.5 * max(val_r2, 0.0) + 0.5 * val_auc

            checkpoint_path = os.path.join(
                self.checkpoint_dir, self.checkpoint_name
            )

            if composite > self._best_score:
                self._best_score = composite
                self._best_meta  = {
                    'epoch'          : epoch,
                    'val_r2'         : val_r2,
                    'val_auc'        : val_auc,
                    'composite'      : composite,
                    'checkpoint_path': checkpoint_path,
                }
                torch.save({
                    'epoch'            : epoch,
                    'model_state_dict' : self.model.state_dict(),
                    'optimizer_state'  : self.optimizer.state_dict(),
                    'val_r2'           : val_r2,
                    'val_auc'          : val_auc,
                    'composite'        : composite,
                }, checkpoint_path)

                if epoch % self.log_every == 0 or epoch == 1:
                    print(f"  --> Checkpoint saved | R²={val_r2:.4f} | AUC={val_auc:.4f}")

            # ── Logging ───────────────────────────────────────────────
            if epoch % self.log_every == 0 or epoch == 1:
                self._log(epoch, train_log, val_r2, val_auc, composite)

        print("=" * len(header))
        print(f"Training complete. Best composite={self._best_score:.4f} "
              f"at epoch {self._best_meta.get('epoch','?')}")
        return self._best_meta

    # ── Training epoch ────────────────────────────────────────────────────

    def _train_epoch(self, X, y, lengths, epoch, N):
        """
        One full pass over the training data in random mini-batches.

        Key term — torch.randperm: returns a random permutation of integers
                   0..N-1. Using this as indices into X and y shuffles the
                   dataset each epoch without copying the data.

        Returns
        ───────
        avg_log : dict  — averaged loss components across all batches
        """
        self.model.train()

        # Random permutation of sample indices for this epoch
        perm = torch.randperm(N)

        # Accumulators for logging — we average over batches at the end
        total_log    = {}
        num_batches  = 0

        for start in range(0, N, self.batch_size):
            idx = perm[start : start + self.batch_size]

            # ── Build batch ───────────────────────────────────────────
            batch_x, batch_y, batch_lens = self._build_batch(X, y, lengths, idx)

            # ── Forward ───────────────────────────────────────────────
            self.optimizer.zero_grad()
            outputs = self.model(batch_x, batch_lens)

            # ── Loss via model's own compute_loss ─────────────────────
            loss, log = self.model.compute_loss(
                outputs, batch_y, batch_lens, epoch
            )

            # ── Backward ──────────────────────────────────────────────
            loss.backward()

            # Gradient clipping — prevents large gradient spikes from
            # destabilising the operator parameters.
            # Key term — clip_grad_norm_: rescales all gradients so their
            #            combined L2 norm does not exceed max_norm.
            #            This does not zero gradients — it only rescales them
            #            when they exceed the threshold.
            nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=1.0)

            self.optimizer.step()

            # ── Post-step hook ────────────────────────────────────────
            # For Koopman: projects K_pheno back to unitary manifold.
            # For GRU: no-op.
            self.model.post_step_hook()

            # ── Accumulate log ────────────────────────────────────────
            for k, v in log.items():
                total_log[k] = total_log.get(k, 0.0) + v
            num_batches += 1

        # Average across batches
        avg_log = {k: v / num_batches for k, v in total_log.items()}
        return avg_log

    # ── Validation ────────────────────────────────────────────────────────

    def _validate(self, X_val, y_val, lengths):
        """
        Stratified balanced validation — matches the original Koopman loop.

        Stratified: equal number of positive and negative samples.
        Balanced: minority class count determines sample size.
        Deterministic: same seed each epoch for stable metric tracking.

        Key term — stratified sampling: sampling strategy that preserves
                   the class ratio. Here we go further and balance the
                   classes to prevent majority-class AUC inflation.

        Returns
        ───────
        val_r2, val_auc : float, float
        """
        self.model.eval()

        with torch.no_grad():
            # ── Stratified balanced index selection ───────────────────
            labels_np = y_val.numpy()
            all_idx   = np.arange(len(labels_np))

            pos_idx = all_idx[labels_np == 1]
            neg_idx = all_idx[labels_np == 0]

            min_count = min(len(pos_idx), len(neg_idx))

            # Fixed seed for reproducibility across epochs
            rng           = np.random.default_rng(42)
            sel_pos       = rng.choice(pos_idx, min_count, replace=False)
            sel_neg       = rng.choice(neg_idx, min_count, replace=False)
            val_idx       = np.concatenate([sel_pos, sel_neg])

            # ── Build validation batch ────────────────────────────────
            val_x, val_y, val_lens = self._build_batch(
                X_val, y_val, lengths, val_idx
            )

            # ── Forward pass ──────────────────────────────────────────
            outputs = self.model(val_x, val_lens)

            # outputs[0] is z_kin for Koopman, z_seq for GRU
            # outputs[-1] is always cls_logits for both models
            z_first    = outputs[0]
            cls_logits = outputs[-1]

            # R² on the first latent output
            # For Koopman: z_kin (kinetic subspace, 16D)
            # For GRU:     z_seq (full latent, 32D)
            val_r2  = compute_val_r2(self.model, z_first, val_lens)
            val_auc = compute_val_auc(cls_logits, val_y)

        return val_r2, val_auc

    # ── Batch builder ─────────────────────────────────────────────────────

    def _build_batch(self, X, y, lengths, idx):
        """
        Slices a mini-batch from pre-converted tensors and moves to device.

        Uses build_velocity_batch convention: idx can be a tensor or array.

        Key term — .to(device): moves a tensor to the specified device
                   (CPU or CUDA). If the tensor is already on that device,
                   this is a no-op with no memory copy.
        """
        idx_list = idx.tolist() if hasattr(idx, 'tolist') else list(idx)

        batch_x   = X[idx_list].to(self.device)
        batch_y   = y[idx_list].to(self.device)
        batch_len = [lengths[i] for i in idx_list]

        return batch_x, batch_y, batch_len

    # ── Logging helpers ───────────────────────────────────────────────────

    def _make_header(self):
        return (f"{'Epoch':>6}  {'Loss':>8}  {'l_dyn':>8}  "
                f"{'l_cls':>8}  {'Val R²':>8}  {'Val AUC':>8}  {'Composite':>10}")

    def _log(self, epoch, log, val_r2, val_auc, composite):
        loss  = log.get('loss',  float('nan'))
        l_dyn = log.get('l_dyn', float('nan'))
        l_cls = log.get('l_cls', float('nan'))
        print(f"{epoch:>6}  {loss:>8.4f}  {l_dyn:>8.5f}  "
              f"{l_cls:>8.4f}  {val_r2:>8.4f}  {val_auc:>8.4f}  {composite:>10.4f}")


print("Trainer defined.")

In [ ]:
# ── Cell E: Usage ─────────────────────────────────────────────────────────
# This replaces both the Koopman and GRU training loops entirely.
# Run Cells A, B, C, D first.

# ── Koopman ───────────────────────────────────────────────────────────────
koopman_model     = AnchoredBlockDiagonalKoopmanNet(input_dim=36, latent_dim=32)
koopman_optimizer = torch.optim.AdamW(
    koopman_model.parameters(), lr=1e-3, weight_decay=1e-4
)

koopman_trainer = Trainer(
    model           = koopman_model,
    optimizer       = koopman_optimizer,
    checkpoint_dir  = DIRS["checkpoints"],
    checkpoint_name = "koopman_trainer_best_2.pt",
    epochs          = 100,
    batch_size      = 16,
    device          = 'cpu',
    log_every       = 10,
)

koopman_best = koopman_trainer.fit(train_split, test_split)
print(koopman_best)

# ── GRU ───────────────────────────────────────────────────────────────────
gru_model     = BlackBoxGraphDynamicsNet(input_dim=72, latent_dim=32)
gru_optimizer = torch.optim.AdamW(
    gru_model.parameters(), lr=1e-3, weight_decay=1e-4
)

gru_trainer = Trainer(
    model           = gru_model,
    optimizer       = gru_optimizer,
    checkpoint_dir  = DIRS["checkpoints"],
    checkpoint_name = "gru_trainer_best_2.pt",
    epochs          = 100,
    batch_size      = 16,
    device          = 'cpu',
    log_every       = 10,
)

gru_best = gru_trainer.fit(train_split, test_split)
print(gru_best)

In [ ]:
evaluator = KoopmanEvaluator(koopman_model, gru_model)
results = evaluator.run(test_split)
evaluator.print_summary(results)
evaluator.plot(results)

In [ ]:
# ── GRU transition diagnostic ─────────────────────────────────────────────
# Check whether the GRU is learning a near-identity transition
# by measuring how much z changes after one autonomous step.

gru_model.eval()
with torch.no_grad():
    # Take first batch from test_split
    sample_x = torch.from_numpy(test_split.X[:16]).float()
    sample_lens = test_split.lengths[:16]

    z_seq, _ = gru_model(sample_x, sample_lens)

    # One autonomous step from each z_t
    z_t     = z_seq[:, :-1].reshape(-1, gru_model.latent_dim)
    gru_inp = torch.zeros_like(z_t)
    z_next  = gru_model.rnn_transition(gru_inp, z_t)

    # How much did z change?
    change = (z_next - z_t).norm(dim=1) / (z_t.norm(dim=1) + 1e-6)
    print(f"Mean relative change per GRU step: {change.mean().item():.4f}")
    print(f"(If close to 0: GRU learned near-identity transition)")
    print(f"(If > 0.05: GRU is actually transforming the state)")

In [ ]:
# ── Cell E (Aspirin): Train both models on rMD17 Aspirin ──────────────────
#
# Dimension contract (mirrors the original ethanol Cell E):
#
#   AnchoredBlockDiagonalKoopmanNet(input_dim = flat_dim)
#       → internally computes phase_space_dim = input_dim * 2
#   BlackBoxGraphDynamicsNet(input_dim = phase_space_dim)
#       → receives the already-doubled dimension
#
#   aspirin has 21 atoms:
#     flat_dim        = 21*20//2  = 210   ← Koopman input_dim
#     phase_space_dim = 2 * 210   = 420   ← adapter.input_dim, GRU input_dim
#     hidden_dim      = 128               (wider encoder for aspirin)
#     latent_dim      = 32,  sub_dim = 16 (unchanged)
#
# NOTE: koopman_compute_loss() uses .view(-1, 16) which matches sub_dim=16.
# If you change latent_dim, update that view accordingly.

import os, torch, numpy as np

# ── 1. Load aspirin data ───────────────────────────────────────────────────
aspirin_adapter = MD17Adapter(
    path         = os.path.join(abhilash437_md17_aspirin_path, 'rmd17_aspirin.npz'),
    molecule     = "aspirin",
    sub_sampling = 2,
    window_len   = 150,
    train_frac   = 0.8,
)
aspirin_train, aspirin_test = aspirin_adapter.load()

flat_dim        = aspirin_adapter.input_dim // 2   # 210
phase_space_dim = aspirin_adapter.input_dim         # 420

print(f"flat_dim           : {flat_dim}")
print(f"phase_space_dim    : {phase_space_dim}")
print(f"train X shape      : {aspirin_train.X.shape}")
print(f"test  X shape      : {aspirin_test.X.shape}")

HIDDEN_DIM = 128
LATENT_DIM = 32   # sub_dim = 16

# ── 3. Koopman model ───────────────────────────────────────────────────────
#   input_dim = flat_dim = 210
#   Inside the net: phase_space_dim = 210 * 2 = 420  ✓ matches X.shape[-1]
koopman_model_asp = AnchoredBlockDiagonalKoopmanNet(
    input_dim  = flat_dim,     # 210  ← NOT adapter.input_dim
    latent_dim = LATENT_DIM,
    hidden_dim = HIDDEN_DIM,
)
koopman_optimizer_asp = torch.optim.AdamW(
    koopman_model_asp.parameters(), lr=5e-4, weight_decay=1e-4
)
n_koop = sum(p.numel() for p in koopman_model_asp.parameters() if p.requires_grad)
print(f"Koopman parameters : {n_koop:,}")

koopman_trainer_asp = Trainer(
    model           = koopman_model_asp,
    optimizer       = koopman_optimizer_asp,
    checkpoint_dir  = DIRS["checkpoints"],
    checkpoint_name = "koopman_aspirin_best.pt",
    epochs          = 150,
    batch_size      = 16,
    device          = DEVICE,
    log_every       = 10,
)
koopman_best_asp = koopman_trainer_asp.fit(aspirin_train, aspirin_test)
print(koopman_best_asp)

# ── 4. GRU baseline ────────────────────────────────────────────────────────
#   input_dim = phase_space_dim = 420
#   BlackBoxGraphDynamicsNet takes the full phase-space width directly
gru_model_asp = BlackBoxGraphDynamicsNet(
    input_dim  = phase_space_dim,   # 420  ← adapter.input_dim
    latent_dim = LATENT_DIM,
    hidden_dim = HIDDEN_DIM,
)
gru_optimizer_asp = torch.optim.AdamW(
    gru_model_asp.parameters(), lr=5e-4, weight_decay=1e-4
)
n_gru = sum(p.numel() for p in gru_model_asp.parameters() if p.requires_grad)
print(f"GRU parameters     : {n_gru:,}")

gru_trainer_asp = Trainer(
    model           = gru_model_asp,
    optimizer       = gru_optimizer_asp,
    checkpoint_dir  = DIRS["checkpoints"],
    checkpoint_name = "gru_aspirin_best.pt",
    epochs          = 150,
    batch_size      = 16,
    device          = DEVICE,
    log_every       = 10,
)
gru_best_asp = gru_trainer_asp.fit(aspirin_train, aspirin_test)
print(gru_best_asp)

In [ ]:
# ── Evaluation: Aspirin ───────────────────────────────────────────────────
import os

# ── 1. Reload best checkpoints (Trainer saves best but trains to the end) ──
koopman_ckpt = torch.load(
    os.path.join(DIRS["checkpoints"], "koopman_aspirin_best.pt"),
    weights_only=False
)
koopman_model_asp.load_state_dict(koopman_ckpt["model_state_dict"])
print(f"Koopman best  →  epoch {koopman_ckpt['epoch']}  |  "
      f"R²={koopman_ckpt['val_r2']:.4f}  |  AUC={koopman_ckpt['val_auc']:.4f}  |  "
      f"composite={koopman_ckpt['composite']:.4f}")

gru_ckpt = torch.load(
    os.path.join(DIRS["checkpoints"], "gru_aspirin_best.pt"),
    weights_only=False
)
gru_model_asp.load_state_dict(gru_ckpt["model_state_dict"])
print(f"GRU     best  →  epoch {gru_ckpt['epoch']}  |  "
      f"R²={gru_ckpt['val_r2']:.4f}  |  AUC={gru_ckpt['val_auc']:.4f}  |  "
      f"composite={gru_ckpt['composite']:.4f}")

# ── 2. Run evaluator ───────────────────────────────────────────────────────
aspirin_evaluator = KoopmanEvaluator(
    koopman_model  = koopman_model_asp,
    baseline_model = gru_model_asp,
    device         = DEVICE,
    rollout_steps  = 29,
    sub_dim        = 16,     # latent_dim // 2
    batch_size     = 64,
)

aspirin_results = aspirin_evaluator.run(aspirin_test)

# ── 3. Print + plot ────────────────────────────────────────────────────────
aspirin_evaluator.print_summary(aspirin_results)
aspirin_evaluator.plot(
    aspirin_results,
    save_path=os.path.join(DIRS["results"], "aspirin_tradeoff.png")
)


In [ ]:
# ── Cell F (Malonaldehyde): Train and Evaluate both models ─────────────────

malon_adapter = MD17Adapter(
    path         = os.path.join(abhilash437_md17_malonaldehyde_path, "rmd17_malonaldehyde.npz"),
    molecule     = "malonaldehyde",
    sub_sampling = 2,
    window_len   = 150,
    train_frac   = 0.8,
)
malon_train, malon_test = malon_adapter.load()

flat_dim        = malon_adapter.input_dim // 2   # 36
phase_space_dim = malon_adapter.input_dim         # 72

print(f"flat_dim           : {flat_dim}")
print(f"phase_space_dim    : {phase_space_dim}")
print(f"train X shape      : {malon_train.X.shape}")
print(f"test  X shape      : {malon_test.X.shape}")

HIDDEN_DIM = 64
LATENT_DIM = 32

# ── 4. Koopman model ───────────────────────────────────────────────────────
koopman_model_malon = AnchoredBlockDiagonalKoopmanNet(
    input_dim  = flat_dim,     # 36
    latent_dim = LATENT_DIM,
    hidden_dim = HIDDEN_DIM,
)
koopman_optimizer_malon = torch.optim.AdamW(
    koopman_model_malon.parameters(), lr=1e-3, weight_decay=1e-4
)
n_koop = sum(p.numel() for p in koopman_model_malon.parameters() if p.requires_grad)
print(f"Koopman parameters : {n_koop:,}")

koopman_trainer_malon = Trainer(
    model           = koopman_model_malon,
    optimizer       = koopman_optimizer_malon,
    checkpoint_dir  = DIRS["checkpoints"],
    checkpoint_name = "koopman_malon_best.pt",
    epochs          = 100,
    batch_size      = 16,
    device          = DEVICE,
    log_every       = 10,
)
koopman_best_malon = koopman_trainer_malon.fit(malon_train, malon_test)
print(koopman_best_malon)

# ── 5. GRU baseline ────────────────────────────────────────────────────────
gru_model_malon = BlackBoxGraphDynamicsNet(
    input_dim  = phase_space_dim,   # 72
    latent_dim = LATENT_DIM,
    hidden_dim = HIDDEN_DIM,
)
gru_optimizer_malon = torch.optim.AdamW(
    gru_model_malon.parameters(), lr=1e-3, weight_decay=1e-4
)
n_gru = sum(p.numel() for p in gru_model_malon.parameters() if p.requires_grad)
print(f"GRU parameters     : {n_gru:,}")

gru_trainer_malon = Trainer(
    model           = gru_model_malon,
    optimizer       = gru_optimizer_malon,
    checkpoint_dir  = DIRS["checkpoints"],
    checkpoint_name = "gru_malon_best.pt",
    epochs          = 100,
    batch_size      = 16,
    device          = DEVICE,
    log_every       = 10,
)
gru_best_malon = gru_trainer_malon.fit(malon_train, malon_test)
print(gru_best_malon)

# ── 6. Evaluation: Malonaldehyde ───────────────────────────────────────────
koopman_ckpt = torch.load(
    os.path.join(DIRS["checkpoints"], "koopman_malon_best.pt"),
    weights_only=False
)
koopman_model_malon.load_state_dict(koopman_ckpt["model_state_dict"])
print(f"Koopman best  →  epoch {koopman_ckpt['epoch']}  |  "
      f"R²={koopman_ckpt['val_r2']:.4f}  |  AUC={koopman_ckpt['val_auc']:.4f}  |  "
      f"composite={koopman_ckpt['composite']:.4f}")

gru_ckpt = torch.load(
    os.path.join(DIRS["checkpoints"], "gru_malon_best.pt"),
    weights_only=False
)
gru_model_malon.load_state_dict(gru_ckpt["model_state_dict"])
print(f"GRU     best  →  epoch {gru_ckpt['epoch']}  |  "
      f"R²={gru_ckpt['val_r2']:.4f}  |  AUC={gru_ckpt['val_auc']:.4f}  |  "
      f"composite={gru_ckpt['composite']:.4f}")

malon_evaluator = KoopmanEvaluator(
    koopman_model  = koopman_model_malon,
    baseline_model = gru_model_malon,
    device         = DEVICE,
    rollout_steps  = 29,
    sub_dim        = 16,
    batch_size     = 64,
)

malon_results = malon_evaluator.run(malon_test)

malon_evaluator.print_summary(malon_results)
malon_evaluator.plot(
    malon_results,
    save_path=os.path.join(DIRS["results"], "malonaldehyde_tradeoff.png")
)


In [ ]:
# ── Cell G (Diagnostic): Verify GRU Collapse / "Dying" vs Koopman Liveliness ──
import matplotlib.pyplot as plt
import torch
import numpy as np
from sklearn.decomposition import PCA

# ── 1. Rollout both models for 100 steps ──────────────────────────────────────
# (Longer horizon to clearly show the asymptotic behavior)
steps = 100
with torch.no_grad():
    # Convert test set to tensor
    X_test_t = torch.tensor(malon_test.X, dtype=torch.float32, device=DEVICE)

    # ── Encode starting states ────────────────────────────────────────────────
    # Koopman initial state: concat(z_kin, z_pheno)
    z_kin_init = koopman_model_malon.encoder_kin(X_test_t[:, 0])
    z_pheno_init = koopman_model_malon.encoder_pheno(X_test_t[:, 0])
    z_kin_init = koopman_model_malon.kin_norm(z_kin_init)
    z_pheno_init = koopman_model_malon.pheno_norm(z_pheno_init)
    z_koop_init = torch.cat([z_kin_init, z_pheno_init], dim=-1) # (B, 32)

    # GRU initial state
    z_gru_init = gru_model_malon.encoder(X_test_t[:, 0]) # (B, 32)

    # ── Perform rollouts ──────────────────────────────────────────────────────
    # Koopman rollout: z_t = z_0 @ K^t^T
    K_koop = koopman_model_malon.K
    z_koop_rollout = []
    for t in range(steps):
        K_pow = torch.matrix_power(K_koop, t)
        z_t = torch.matmul(z_koop_init, K_pow.t())
        z_koop_rollout.append(z_t)
    z_koop_rollout = torch.stack(z_koop_rollout, dim=1).cpu().numpy() # (B, steps, 32)

    # GRU rollout: using forward_rollout
    z_gru_rollout = gru_model_malon.forward_rollout(
        z_gru_init.unsqueeze(1), steps=steps, latent_seed=True
    ).cpu().numpy() # (B, steps, 32)

# ── 2. Compute Diagnostics ────────────────────────────────────────────────────
# Metric A: Step-to-step velocity ||z_{t+1} - z_t|| (averaged over batch)
# If this goes to 0, the state has stopped moving (fixed point)
koop_velocity = np.mean(np.linalg.norm(z_koop_rollout[:, 1:] - z_koop_rollout[:, :-1], axis=-1), axis=0)
gru_velocity = np.mean(np.linalg.norm(z_gru_rollout[:, 1:] - z_gru_rollout[:, :-1], axis=-1), axis=0)

# Metric B: Cross-sequence variance (variance of states across the batch)
# If this goes to 0, all sequences have collapsed to the EXACT same point
koop_cross_var = np.mean(np.var(z_koop_rollout, axis=0), axis=-1)
gru_cross_var = np.mean(np.var(z_gru_rollout, axis=0), axis=-1)

# ── 3. Plot results ───────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Plot 1: Step-to-Step Velocity
axes[0, 0].plot(koop_velocity, label="Koopman", color="#2166ac", lw=2)
axes[0, 0].plot(gru_velocity, label="GRU Baseline", color="#d6604d", lw=2)
axes[0, 0].set_title("Step-to-Step Velocity ($||z_{t+1} - z_t||_2$)", fontsize=12)
axes[0, 0].set_xlabel("Rollout Step")
axes[0, 0].set_ylabel("Mean Step Norm")
axes[0, 0].grid(True, linestyle="--", alpha=0.6)
axes[0, 0].legend()

# Plot 2: Cross-Sequence Variance
axes[0, 1].plot(koop_cross_var, label="Koopman", color="#2166ac", lw=2)
axes[0, 1].plot(gru_cross_var, label="GRU Baseline", color="#d6604d", lw=2)
axes[0, 1].set_title("Cross-Sequence Variance (State diversity across batch)", fontsize=12)
axes[0, 1].set_xlabel("Rollout Step")
axes[0, 1].set_ylabel("Mean Variance")
axes[0, 1].grid(True, linestyle="--", alpha=0.6)
axes[0, 1].legend()

# Plot 3: 2D Trajectory Projection of first sequence (PCA)
pca_koop = PCA(n_components=2)
z_koop_pca = pca_koop.fit_transform(z_koop_rollout[0])
axes[1, 0].plot(z_koop_pca[:, 0], z_koop_pca[:, 1], color="#2166ac", marker="o", markersize=3, alpha=0.7)
axes[1, 0].scatter(z_koop_pca[0, 0], z_koop_pca[0, 1], color="green", s=100, label="Start", zorder=5)
axes[1, 0].scatter(z_koop_pca[-1, 0], z_koop_pca[-1, 1], color="red", s=100, label="End", zorder=5)
axes[1, 0].set_title("Koopman Latent Trajectory (PCA projection)", fontsize=12)
axes[1, 0].set_xlabel("PC 1")
axes[1, 0].set_ylabel("PC 2")
axes[1, 0].grid(True, linestyle="--", alpha=0.6)
axes[1, 0].legend()

# Plot 4: GRU Trajectory Projection
pca_gru = PCA(n_components=2)
z_gru_pca = pca_gru.fit_transform(z_gru_rollout[0])
axes[1, 1].plot(z_gru_pca[:, 0], z_gru_pca[:, 1], color="#d6604d", marker="o", markersize=3, alpha=0.7)
axes[1, 1].scatter(z_gru_pca[0, 0], z_gru_pca[0, 1], color="green", s=100, label="Start", zorder=5)
axes[1, 1].scatter(z_gru_pca[-1, 0], z_gru_pca[-1, 1], color="red", s=100, label="End", zorder=5)
axes[1, 1].set_title("GRU Latent Trajectory (PCA projection)", fontsize=12)
axes[1, 1].set_xlabel("PC 1")
axes[1, 1].set_ylabel("PC 2")
axes[1, 1].grid(True, linestyle="--", alpha=0.6)
axes[1, 1].legend()

plt.tight_layout()
plt.show()


In [ ]:
gru_reg_model_malon = RegularizedGRUBaselineNet(
    input_dim  = phase_space_dim,   # 72
    latent_dim = LATENT_DIM,
    hidden_dim = HIDDEN_DIM,
    beta       = 1.0,               # Strong norm preservation penalty
)
gru_reg_optimizer_malon = torch.optim.AdamW(
    gru_reg_model_malon.parameters(), lr=1e-3, weight_decay=1e-4
)
n_gru_reg = sum(p.numel() for p in gru_reg_model_malon.parameters() if p.requires_grad)
print(f"Regularized GRU parameters : {n_gru_reg:,}")

gru_reg_trainer_malon = Trainer(
    model           = gru_reg_model_malon,
    optimizer       = gru_reg_optimizer_malon,
    checkpoint_dir  = DIRS["checkpoints"],
    checkpoint_name = "gru_reg_malon_best.pt",
    epochs          = 100,
    batch_size      = 16,
    device          = DEVICE,
    log_every       = 10,
)
gru_reg_best_malon = gru_reg_trainer_malon.fit(malon_train, malon_test)
print(gru_reg_best_malon)


In [ ]:
# 1. Load Koopman best
koopman_ckpt = torch.load(os.path.join(DIRS["checkpoints"], "koopman_malon_best.pt"), weights_only=False)
koopman_model_malon.load_state_dict(koopman_ckpt["model_state_dict"])

# 2. Load Unconstrained GRU best
gru_ckpt = torch.load(os.path.join(DIRS["checkpoints"], "gru_malon_best.pt"), weights_only=False)
gru_model_malon.load_state_dict(gru_ckpt["model_state_dict"])

# 3. Load Regularized GRU best
gru_reg_ckpt = torch.load(os.path.join(DIRS["checkpoints"], "gru_reg_malon_best.pt"), weights_only=False)
gru_reg_model_malon.load_state_dict(gru_reg_ckpt["model_state_dict"])

# Run evaluator for Unconstrained GRU comparison
evaluator_unconstrained = KoopmanEvaluator(
    koopman_model  = koopman_model_malon,
    baseline_model = gru_model_malon,
    device         = DEVICE,
    rollout_steps  = 29,
    sub_dim        = 16,
    batch_size     = 64,
)
res_unconstrained = evaluator_unconstrained.run(malon_test)

# Run evaluator for Regularized GRU comparison
evaluator_regularized = KoopmanEvaluator(
    koopman_model  = koopman_model_malon,
    baseline_model = gru_reg_model_malon,
    device         = DEVICE,
    rollout_steps  = 29,
    sub_dim        = 16,
    batch_size     = 64,
)
res_regularized = evaluator_regularized.run(malon_test)

# Print 3-Way Summary Table
sep = "=" * 90
print(sep)
print(f"3-WAY COMPARISON SUMMARY — {res_unconstrained.meta.get('dataset','?')} {res_unconstrained.meta.get('molecule','')}")
print(sep)
print(f"  {'Step':>4}  |  {'Koopman MSE':>12}  |  {'Unconstrained GRU':>18}  |  {'Regularized GRU':>16}")
print("-" * 90)
for s in range(len(res_unconstrained.koopman_mse_mean)):
    print(f"  {s+1:>4}  |  "
          f"{res_unconstrained.koopman_mse_mean[s]:>12.4f}  |  "
          f"{res_unconstrained.baseline_mse_mean[s]:>18.4f}  |  "
          f"{res_regularized.baseline_mse_mean[s]:>16.4f}")
print("-" * 90)
print("GEOMETRY RETENTION @ final step:")
print(f"  Koopman          : {res_unconstrained.koopman_geom_mean[-1]:.4f}")
print(f"  Unconstrained GRU: {res_unconstrained.baseline_geom_mean[-1]:.4f}")
print(f"  Regularized GRU  : {res_regularized.baseline_geom_mean[-1]:.4f}")
print("-" * 90)
print("COLLAPSE DIAGNOSTIC (Relative Change):")
print(f"  Koopman          : {res_unconstrained.relative_change_koopman:.4f}")
print(f"  Unconstrained GRU: {res_unconstrained.relative_change_baseline:.4f}")
print(f"  Regularized GRU  : {res_regularized.relative_change_baseline:.4f}")
print(sep)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

steps = 100
X_test_t = torch.tensor(malon_test.X, dtype=torch.float32, device=DEVICE)

# Set models to eval mode
koopman_model_malon.eval()
gru_model_malon.eval()
gru_reg_model_malon.eval()

with torch.no_grad():
    # 1. Koopman rollout
    z_kin_init = koopman_model_malon.encoder_kin(X_test_t[:, 0])
    z_pheno_init = koopman_model_malon.encoder_pheno(X_test_t[:, 0])
    z_kin_init = koopman_model_malon.kin_norm(z_kin_init)
    z_pheno_init = koopman_model_malon.pheno_norm(z_pheno_init)
    z_koop_init = torch.cat([z_kin_init, z_pheno_init], dim=-1) # (B, 32)

    K_koop = koopman_model_malon.K
    z_koop_rollout = []
    for t in range(steps):
        K_pow = torch.matrix_power(K_koop, t)
        z_t = torch.matmul(z_koop_init, K_pow.t())
        z_koop_rollout.append(z_t)
    z_koop_rollout = torch.stack(z_koop_rollout, dim=1).cpu().numpy() # (B, steps, 32)

    # 2. Unconstrained GRU rollout
    z_gru_init = gru_model_malon.encoder(X_test_t[:, 0]) # (B, 32)
    z_gru_rollout = gru_model_malon.forward_rollout(
        z_gru_init.unsqueeze(1), steps=steps, latent_seed=True
    ).cpu().numpy() # (B, steps, 32)

    # 3. Regularized GRU rollout
    z_gru_reg_init = gru_reg_model_malon.encoder(X_test_t[:, 0]) # (B, 32)
    z_gru_reg_rollout = gru_reg_model_malon.forward_rollout(
        z_gru_reg_init.unsqueeze(1), steps=steps, latent_seed=True
    ).cpu().numpy() # (B, steps, 32)

# Compute step-to-step velocity ||z_{t+1} - z_t||_2 (averaged over batch)
koop_velocity = np.mean(np.linalg.norm(z_koop_rollout[:, 1:] - z_koop_rollout[:, :-1], axis=-1), axis=0)
gru_velocity = np.mean(np.linalg.norm(z_gru_rollout[:, 1:] - z_gru_rollout[:, :-1], axis=-1), axis=0)
gru_reg_velocity = np.mean(np.linalg.norm(z_gru_reg_rollout[:, 1:] - z_gru_reg_rollout[:, :-1], axis=-1), axis=0)

# Compute cross-sequence variance (state diversity across batch)
koop_cross_var = np.mean(np.var(z_koop_rollout, axis=0), axis=-1)
gru_cross_var = np.mean(np.var(z_gru_rollout, axis=0), axis=-1)
gru_reg_cross_var = np.mean(np.var(z_gru_reg_rollout, axis=0), axis=-1)

# Compute average norm over time
koop_norm = np.mean(np.linalg.norm(z_koop_rollout, axis=-1), axis=0)
gru_norm = np.mean(np.linalg.norm(z_gru_rollout, axis=-1), axis=0)
gru_reg_norm = np.mean(np.linalg.norm(z_gru_reg_rollout, axis=-1), axis=0)

# Create 2x3 Plotting Dashboard
fig, axes = plt.subplots(2, 3, figsize=(20, 12))

# Row 1, Col 1: Step-to-Step Velocity
axes[0, 0].plot(koop_velocity, label="Koopman", color="#2166ac", lw=2)
axes[0, 0].plot(gru_velocity, label="Unconstrained GRU", color="#d6604d", lw=2)
axes[0, 0].plot(gru_reg_velocity, label="Regularized GRU (norm-pres)", color="#4daf4a", lw=2)
axes[0, 0].set_title("Step-to-Step Velocity ($||z_{t+1} - z_t||_2$)", fontsize=12)
axes[0, 0].set_xlabel("Rollout Step")
axes[0, 0].set_ylabel("Mean Step Norm")
axes[0, 0].grid(True, linestyle="--", alpha=0.6)
axes[0, 0].legend()

# Row 1, Col 2: Cross-Sequence Variance
axes[0, 1].plot(koop_cross_var, label="Koopman", color="#2166ac", lw=2)
axes[0, 1].plot(gru_cross_var, label="Unconstrained GRU", color="#d6604d", lw=2)
axes[0, 1].plot(gru_reg_cross_var, label="Regularized GRU (norm-pres)", color="#4daf4a", lw=2)
axes[0, 1].set_title("Cross-Sequence Variance (State diversity)", fontsize=12)
axes[0, 1].set_xlabel("Rollout Step")
axes[0, 1].set_ylabel("Mean Variance")
axes[0, 1].grid(True, linestyle="--", alpha=0.6)
axes[0, 1].legend()

# Row 1, Col 3: Mean Latent Norm
axes[0, 2].plot(koop_norm, label="Koopman", color="#2166ac", lw=2)
axes[0, 2].plot(gru_norm, label="Unconstrained GRU", color="#d6604d", lw=2)
axes[0, 2].plot(gru_reg_norm, label="Regularized GRU (norm-pres)", color="#4daf4a", lw=2)
axes[0, 2].set_title("Mean Latent Vector Norm ($||z_t||_2$)", fontsize=12)
axes[0, 2].set_xlabel("Rollout Step")
axes[0, 2].set_ylabel("Mean L2 Norm")
axes[0, 2].grid(True, linestyle="--", alpha=0.6)
axes[0, 2].legend()

# Row 2, Col 1: Koopman Trajectory (PCA)
pca_koop = PCA(n_components=2)
z_koop_pca = pca_koop.fit_transform(z_koop_rollout[0])
axes[1, 0].plot(z_koop_pca[:, 0], z_koop_pca[:, 1], color="#2166ac", marker="o", markersize=3, alpha=0.7)
axes[1, 0].scatter(z_koop_pca[0, 0], z_koop_pca[0, 1], color="green", s=100, label="Start", zorder=5)
axes[1, 0].scatter(z_koop_pca[-1, 0], z_koop_pca[-1, 1], color="red", s=100, label="End", zorder=5)
axes[1, 0].set_title("Koopman Latent Trajectory (PCA)", fontsize=12)
axes[1, 0].set_xlabel("PC 1")
axes[1, 0].set_ylabel("PC 2")
axes[1, 0].grid(True, linestyle="--", alpha=0.6)
axes[1, 0].legend()

# Row 2, Col 2: Unconstrained GRU Trajectory (PCA)
pca_gru = PCA(n_components=2)
z_gru_pca = pca_gru.fit_transform(z_gru_rollout[0])
axes[1, 1].plot(z_gru_pca[:, 0], z_gru_pca[:, 1], color="#d6604d", marker="o", markersize=3, alpha=0.7)
axes[1, 1].scatter(z_gru_pca[0, 0], z_gru_pca[0, 1], color="green", s=100, label="Start", zorder=5)
axes[1, 1].scatter(z_gru_pca[-1, 0], z_gru_pca[-1, 1], color="red", s=100, label="End", zorder=5)
axes[1, 1].set_title("Unconstrained GRU Trajectory (PCA)", fontsize=12)
axes[1, 1].set_xlabel("PC 1")
axes[1, 1].set_ylabel("PC 2")
axes[1, 1].grid(True, linestyle="--", alpha=0.6)
axes[1, 1].legend()

# Row 2, Col 3: Regularized GRU Trajectory (PCA)
pca_gru_reg = PCA(n_components=2)
z_gru_reg_pca = pca_gru_reg.fit_transform(z_gru_reg_rollout[0])
axes[1, 2].plot(z_gru_reg_pca[:, 0], z_gru_reg_pca[:, 1], color="#4daf4a", marker="o", markersize=3, alpha=0.7)
axes[1, 2].scatter(z_gru_reg_pca[0, 0], z_gru_reg_pca[0, 1], color="green", s=100, label="Start", zorder=5)
axes[1, 2].scatter(z_gru_reg_pca[-1, 0], z_gru_reg_pca[-1, 1], color="red", s=100, label="End", zorder=5)
axes[1, 2].set_title("Regularized GRU Trajectory (PCA)", fontsize=12)
axes[1, 2].set_xlabel("PC 1")
axes[1, 2].set_ylabel("PC 2")
axes[1, 2].grid(True, linestyle="--", alpha=0.6)
axes[1, 2].legend()

plt.tight_layout()
plt.show()
